# GeoE and Eagle-I Threshold Analysis

This notebook develops a physically meaningful `space_weather_extreme` threshold by comparing two years of interpolated geoelectric-field exposure against Eagle-I outage impacts. The aim is not minute-level correlation. Eagle-I timestamps may reflect heterogeneous reporting sources and latencies, so the analysis should work with impact episodes and hazard episodes.

## Analysis Concept

The working design is a conditional event-quiet analysis. Instead of treating every outage row as an independent point, identify impact episodes with onset, peak, recovery, geographic footprint, affected assets or customers, and duration. For every impact episode, construct matched quiet periods that resemble the impact period in known confounders such as hour of day, day of week, season, temperature, precipitation, wind, system load if available, antecedent outage level, and regional operating state. The quiet periods should not be matched on GMD/geoelectric activity.

The threshold signal is a distribution shift: compare geoelectric hazard descriptors during impact episodes against matched quiet periods. A candidate threshold becomes interesting when it is disproportionately represented during impact periods rather than merely common in the background.

## Candidate Threshold Quantity

A useful diagnostic is an impact-conditioned hazard anomaly:

$$
A(h) = \log \frac{P(H \ge h \mid I, B)}{P(H \ge h \mid Q, B)}
$$

where $I$ denotes impact episodes, $Q$ denotes matched quiet periods, and $B$ denotes the conditioning background such as season, hour, weekday/weekend, weather, antecedent outage level, and operating context. In practice this can be estimated with binned exceedance rates, bootstrap uncertainty, or a conditional/logistic model.

## Next Implementation Steps

1. ~~Generate or load the two-year interpolated, fill-masked, no-extrapolation geoE product on the analysis grid.~~
2. ~~Map Eagle-I counties to analysis-grid cells or regions, preserving county FIPS as five-character strings.~~
3. Build impact episodes from Eagle-I and matched quiet periods from comparable background windows. Builder cells are implemented and cache-aware; run them only when ready to scan the large Eagle-I files.
4. Compute geoE descriptors for each impact and quiet window across several lag/lead buffers.
5. Estimate threshold candidates using distribution shifts, log exceedance ratios, and bootstrap uncertainty.
6. Promote the resulting threshold only after checking that the candidate is robust to Eagle-I timing uncertainty and background matching choices.

In [20]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay, cKDTree

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ROOT = Path('/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data')
EXTERNAL_DATA_ROOT = ROOT

WILDFIRE_DIR = ROOT / 'wildfire' / 'daily'
TERRESTRIAL_DIR = ROOT / 'terrestrial_weather'
SPACE_WEATHER_DIR = ROOT / 'space_weather' / 'NOAA_geoE'
GRID_PATH = ROOT / 'analysis_grid_CONUS_50km.nc'
if not GRID_PATH.exists():
    GRID_PATH = REPO_ROOT / 'data' / 'analysis_grid_CONUS_50km.nc'

EAGLEI_DIR = Path('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I')
EAGLEI_PROCESSED_DIR = EAGLEI_DIR / 'processed'
EAGLEI_CANDIDATES = sorted(EAGLEI_DIR.glob('*.csv')) if EAGLEI_DIR.exists() else []

RUN_START = pd.Timestamp('2024-01-01')
RUN_END = pd.Timestamp('2025-12-31 23:59:59')
TIME_START = str(RUN_START.date())
TIME_END = str(RUN_END.date())

SPACE_WEATHER_PROCESSED_DIR = ROOT / 'space_weather' / 'processed_data'
GEOE_OUTPUT_DIR = SPACE_WEATHER_PROCESSED_DIR
EAGLEI_MAPPING_DIR = EAGLEI_PROCESSED_DIR
EAGLEI_SPACE_WEATHER_DIR = SPACE_WEATHER_PROCESSED_DIR
OUTPUT_DIR = EAGLEI_SPACE_WEATHER_DIR

GEOE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EAGLEI_MAPPING_DIR.mkdir(parents=True, exist_ok=True)
EAGLEI_SPACE_WEATHER_DIR.mkdir(parents=True, exist_ok=True)

print(f'Space-weather source directory: {SPACE_WEATHER_DIR}  exists={SPACE_WEATHER_DIR.exists()}')
print(f'Analysis grid: {GRID_PATH}  exists={GRID_PATH.exists()}')
print(f'Eagle-I source directory: {EAGLEI_DIR}  csv_files={len(EAGLEI_CANDIDATES)}')
print(f'Eagle-I general processed directory: {EAGLEI_MAPPING_DIR}')
print(f'Space-weather processed output directory: {EAGLEI_SPACE_WEATHER_DIR}')

Space-weather source directory: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE  exists=True
Analysis grid: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/analysis_grid_CONUS_50km.nc  exists=True
Eagle-I source directory: /Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I  csv_files=7
Eagle-I general processed directory: /Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I/processed
Space-weather processed output directory: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/processed_data


In [21]:
def first_existing_path(paths: Iterable[Path]) -> Optional[Path]:
    for path in paths:
        if path.exists():
            return path
    return None


def normalize_county_fips(values: pd.Series) -> pd.Series:
    return values.astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)


def _find_column(columns: Iterable[str], candidates: Iterable[str]) -> Optional[str]:
    lower = {column.lower(): column for column in columns}
    normalized = {column.lower().replace(' ', '_'): column for column in columns}
    for candidate in candidates:
        candidate_lower = candidate.lower()
        candidate_normalized = candidate_lower.replace(' ', '_')
        if candidate_lower in lower:
            return lower[candidate_lower]
        if candidate_normalized in normalized:
            return normalized[candidate_normalized]
    return None


def load_eaglei(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    time_column = _find_column(df.columns, ['Run Start Time', 'run_start_time', 'start_time', 'time', 'timestamp'])
    if time_column is None:
        raise KeyError(f'Could not identify an Eagle-I timestamp column. Columns: {list(df.columns)}')

    fips_column = _find_column(df.columns, ['Fips Code', 'fips_code', 'county_fips', 'geoid', 'fips'])
    if fips_column is None:
        raise KeyError(f'Could not identify an Eagle-I county FIPS column. Columns: {list(df.columns)}')

    outage_column = _find_column(df.columns, ['Customers Out', 'customers_out', 'customers_outaged', 'outages', 'n_outages', 'sum'])
    if outage_column is None:
        numeric_columns = df.select_dtypes(include='number').columns.tolist()
        outage_column = numeric_columns[0] if numeric_columns else None
    if outage_column is None:
        raise KeyError('Could not identify a numeric Eagle-I outage/intensity column.')

    out = df.copy()
    out['time'] = pd.to_datetime(out[time_column], errors='coerce', utc=True).dt.tz_convert(None)
    out['county_fips'] = normalize_county_fips(out[fips_column])
    out['outage_value'] = pd.to_numeric(out[outage_column], errors='coerce')
    out = out.dropna(subset=['time', 'county_fips', 'outage_value'])
    out = out[(out['time'] >= RUN_START) & (out['time'] <= RUN_END)].copy()
    out.attrs['source_path'] = str(path)
    out.attrs['time_column'] = time_column
    out.attrs['fips_column'] = fips_column
    out.attrs['outage_column'] = outage_column
    return out


# Avoid eager loading here: Eagle-I holdings include many large daily files. Downstream cells
# should load only the subset needed for a specific analysis.
eaglei_path = first_existing_path(EAGLEI_CANDIDATES)
eaglei = pd.DataFrame()
print(f'Eagle-I helper functions ready. Candidate top-level CSV files: {len(EAGLEI_CANDIDATES)}')
if eaglei_path is not None:
    header = pd.read_csv(eaglei_path, nrows=0)
    print(f'First candidate: {eaglei_path}')
    print(f'Columns: {list(header.columns)}')

Eagle-I helper functions ready. Candidate top-level CSV files: 7
First candidate: /Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I/20230325T000300_20230409T000400_county_outage_data.csv
Columns: ['County', 'State', 'Fips Code', 'Customers Out', 'Total Customers', 'Total Customers Type', 'Percent Out', 'Run Start Time', 'Contains Override Data']


In [ ]:
# Eagle-I reusable processed-output paths
#
# The county 15-minute panel is a general Eagle-I product and should live with other
# reusable Eagle-I processed files. Space-weather-specific episodes and quiet windows
# continue to use OUTPUT_DIR.

EAGLEI_COUNTY_PANEL_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_15min_panel_2024_2025.csv'
EAGLEI_COUNTY_PANEL_QA_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_15min_panel_qa_2024_2025.csv'

print(f'Eagle-I county panel cache: {EAGLEI_COUNTY_PANEL_CSV}')
print(f'Eagle-I county panel QA:    {EAGLEI_COUNTY_PANEL_QA_CSV}')

In [ ]:
# Build Eagle-I impact episodes from large CSV holdings in chunks
#
# This cell defines the episode-builder functions and panel scan but does not run the full scan unless RUN_EAGLEI_PANEL_BUILD=True.
# Full execution reads hundreds of daily Eagle-I files, so leave the flag off while editing the notebook.
#
# Moved above the threshold-builder cell (2026-09-21): the threshold builder calls
# build_eaglei_county_panel() and eagle_i_csv_files_for_impact_window(), which are defined here.
# They must be defined before that cell runs, so this cell now comes first.
#
# Data-quality fixes (2026-09-21):
# 1. State-name normalization: raw Eagle-I files spell state inconsistently across formats/vintages
#    ("TX" in the daily granular files vs "Texas" in the yearly bulk files). Left unnormalized, the
#    panel groupby key (time, county_fips, county_name, state_abbr) treats these as two different
#    series for the same county, so the same timestamp can carry two parallel rows with different
#    total_customers denominators and never gets deduplicated by the max-aggregation groupby.
# 2. Override-data suppression, not deletion: the daily granular files carry `Total Customers Type`
#    ('collected' vs 'mixed'/estimated) and `Contains Override Data` (true/false) columns. Rows flagged
#    as override or non-collected can carry garbage values injected by Eagle-I's own estimation
#    pipeline during major events (confirmed for Harris County, TX on 2024-05-21: a mixed/override row
#    reported customers_out=875,965,953, ~6,700x the real value). Rather than dropping these rows
#    outright, their numeric fields (customers_out, percent_out) are set to NaN and the row is flagged
#    `is_suppressed_override=True`, but the (time, county_fips) slot itself is kept in the panel. This
#    matters for persistence: if a slot were dropped entirely instead of nulled, a single bad reading
#    in the middle of a real, sustained outage would open up a false gap and could cause a genuine
#    episode to fail the persistence check below for no physical reason. Keeping the slot (with a NaN
#    value) lets the persistence check tell the difference between "Eagle-I reported here but we
#    don't trust the number" and "nothing was reported here at all, possibly because the outage
#    actually cleared." Only the former is bridged; the latter is treated as a real gap.
# 3. Persistence requirement: an episode must contain at least OUTAGE_EPISODE_MIN_CONSECUTIVE_SAMPLES
#    back-to-back 15-minute readings above threshold (no unexplained gap between them), not just
#    readings bridged by the 1-hour merge-gap tolerance. A gap between two active readings is bridged
#    (does not break persistence) only if every intervening 15-minute slot in the panel is present and
#    flagged is_suppressed_override; any other kind of gap (missing report, or a real reading that
#    fell back below threshold) breaks the run. This removes isolated single-reading "blips" that were
#    being stitched into hour-long episodes purely because the gap-merge rule is forgiving, while not
#    penalizing a genuinely sustained outage that happens to have one suppressed bad reading in it.
#
# Restructured for un-cached runs (2026-09-21):
# - EAGLEI_COUNTY_PANEL_CSV no longer gets redefined here to a different path than cell 6's
#   EAGLEI_PROCESSED_DIR definition. That redefinition (to OUTPUT_DIR) was the root cause of two
#   duplicate ~5GB panel caches living in two different directories -- one under
#   EAGLE-I/processed/, one under space_weather/processed_data/.
# - Eagle-I-only outputs (impact episodes, episode QA, episode outliers) now write to
#   EAGLEI_PROCESSED_DIR, not OUTPUT_DIR. OUTPUT_DIR (space_weather/processed_data) is reserved
#   for results from aligning Eagle-I impact episodes against GeoE space-weather data, not for
#   Eagle-I-only analysis products.
# - This cell's execution block now only builds/loads the county panel (renamed
#   RUN_EAGLEI_EPISODE_BUILD -> RUN_EAGLEI_PANEL_BUILD to match). Impact-episode identification
#   moved to the threshold-builder cell below, since it needs the county threshold table that
#   cell produces -- on a fully un-cached run, this cell used to try to read
#   EAGLEI_COUNTY_THRESHOLDS_CSV before the next cell had a chance to create it. The threshold
#   cell reuses this cell's in-memory county_outage_panel instead of loading/rescanning it, so a
#   from-scratch run of both cells in physical top-to-bottom order does exactly one full Eagle-I
#   file scan, not two.

EAGLEI_IMPACT_EPISODES_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_impact_episodes_2024_2025.csv'
EAGLEI_EPISODE_QA_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_impact_episode_qa_2024_2025.csv'
EAGLEI_EPISODE_OUTLIERS_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_impact_episode_outliers_2024_2025.csv'
RUN_EAGLEI_PANEL_BUILD = True

OUTAGE_EPISODE_THRESHOLD_MODE = 'county_p99'
OUTAGE_EPISODE_MIN_CUSTOMERS = 50.0
OUTAGE_EPISODE_MIN_PERCENT_OUT = 0.5
OUTAGE_EPISODE_MAX_GAP = pd.Timedelta('1h')
OUTAGE_EPISODE_MIN_DURATION = pd.Timedelta('30min')
OUTAGE_EPISODE_MIN_CONSECUTIVE_SAMPLES = 2
EAGLEI_PANEL_CADENCE = pd.Timedelta('15min')

# US state/territory full-name -> USPS abbreviation, used to normalize state_abbr so the same
# state spelled two ways in different Eagle-I file vintages collapses to one canonical value.
US_STATE_NAME_TO_ABBR = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'district of columbia': 'DC',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL',
    'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA',
    'maine': 'ME', 'maryland': 'MD', 'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN',
    'mississippi': 'MS', 'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV',
    'new hampshire': 'NH', 'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY',
    'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK', 'oregon': 'OR',
    'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC', 'south dakota': 'SD',
    'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT', 'virginia': 'VA',
    'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY',
    'puerto rico': 'PR', 'guam': 'GU', 'american samoa': 'AS', 'virgin islands': 'VI',
    'northern mariana islands': 'MP',
}


def normalize_state_abbr(values: pd.Series) -> pd.Series:
    """Collapse full state names and USPS abbreviations to a single canonical USPS code.

    Raw Eagle-I files spell state inconsistently: the daily granular files use USPS codes
    ("TX") while the yearly bulk files spell out the full name ("Texas"). Without this
    normalization the panel groupby key treats these as separate series for the same
    county, preventing deduplication and letting duplicate/override rows survive.
    """
    cleaned = values.astype('string').str.strip()
    lowered = cleaned.str.lower()
    mapped = lowered.map(US_STATE_NAME_TO_ABBR)
    is_already_abbr = cleaned.str.len().eq(2)
    result = mapped.where(mapped.notna(), cleaned.where(is_already_abbr, cleaned))
    return result.str.upper()


def eagle_i_csv_files_for_impact_window(start_year: int = 2024, end_year: int = 2025) -> list[Path]:
    files = []
    for path in sorted(EAGLEI_DIR.rglob('*.csv')):
        if EAGLEI_PROCESSED_DIR in path.parents:
            continue
        name = path.name.lower()
        if any(str(year) in name for year in range(start_year, end_year + 1)):
            files.append(path)
    return files


def read_eaglei_panel_chunks(paths: Iterable[Path], chunksize: int = 250_000) -> Iterable[pd.DataFrame]:
    for path in paths:
        header = pd.read_csv(path, nrows=0)
        time_col = _find_column(header.columns, ['Run Start Time', 'run_start_time', 'start_time', 'time', 'timestamp'])
        fips_col = _find_column(header.columns, ['Fips Code', 'fips_code', 'county_fips', 'geoid', 'fips'])
        county_col = _find_column(header.columns, ['County', 'county'])
        state_col = _find_column(header.columns, ['State', 'state'])
        customers_col = _find_column(header.columns, ['Customers Out', 'customers_out', 'customers_outaged', 'outages', 'n_outages'])
        total_col = _find_column(header.columns, ['Total Customers', 'total_customers'])
        percent_col = _find_column(header.columns, ['Percent Out', 'percent_out'])
        customers_type_col = _find_column(header.columns, ['Total Customers Type', 'total_customers_type'])
        override_col = _find_column(header.columns, ['Contains Override Data', 'contains_override_data'])
        if time_col is None or fips_col is None or customers_col is None:
            print(f'Skipping {path.name}: missing required Eagle-I columns')
            continue
        usecols = [time_col, fips_col, customers_col]
        for optional_col in [county_col, state_col, total_col, percent_col, customers_type_col, override_col]:
            if optional_col is not None and optional_col not in usecols:
                usecols.append(optional_col)
        for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
            rename_map = {time_col: 'time', fips_col: 'county_fips', customers_col: 'customers_out'}
            if county_col is not None:
                rename_map[county_col] = 'county_name'
            if state_col is not None:
                rename_map[state_col] = 'state_abbr'
            if total_col is not None:
                rename_map[total_col] = 'total_customers'
            if percent_col is not None:
                rename_map[percent_col] = 'percent_out'
            if customers_type_col is not None:
                rename_map[customers_type_col] = 'total_customers_type'
            if override_col is not None:
                rename_map[override_col] = 'contains_override_data'
            chunk = chunk.rename(columns=rename_map)
            chunk['time'] = pd.to_datetime(chunk['time'], errors='coerce', utc=True).dt.tz_convert(None)
            chunk['county_fips'] = normalize_county_fips(chunk['county_fips'])
            chunk['customers_out'] = pd.to_numeric(chunk['customers_out'], errors='coerce')
            if 'total_customers' not in chunk:
                chunk['total_customers'] = np.nan
            else:
                chunk['total_customers'] = pd.to_numeric(chunk['total_customers'], errors='coerce')
            if 'percent_out' not in chunk:
                chunk['percent_out'] = np.where(chunk['total_customers'] > 0, 100.0 * chunk['customers_out'] / chunk['total_customers'], np.nan)
            else:
                chunk['percent_out'] = pd.to_numeric(chunk['percent_out'], errors='coerce')
            if 'county_name' not in chunk:
                chunk['county_name'] = pd.NA
            if 'state_abbr' not in chunk:
                chunk['state_abbr'] = pd.NA
            else:
                chunk['state_abbr'] = normalize_state_abbr(chunk['state_abbr'])

            # Flag override/estimated or explicitly non-collected rows, and null their numeric
            # fields, rather than dropping the row outright. See the cell-level note above for why
            # this matters for the persistence check. This column only exists in the daily
            # granular file format; files without it get is_suppressed_override=False everywhere.
            #
            # NOTE (2026-09-21, pilot QA finding): 'Total Customers Type' != 'collected'
            # ('mixed'/'modeled') is NOT a data-quality flag -- it is Eagle-I's normal way of
            # describing counties served by multiple utilities, some of which don't report
            # granular data. In the May 2024 pilot window it covered ~71% of all panel rows.
            # Suppressing on that condition nulled out the large majority of legitimate data.
            # The real quality flag -- the one that actually caught the Harris County, TX
            # 6,700x customers_out inflation on 2024-05-21 that motivated this fix -- is
            # 'Contains Override Data' == True, which is rare (e.g. 25 of ~72k rows in the
            # worst single day of the pilot window, concentrated exactly on that storm event).
            # So we suppress ONLY on Contains Override Data == True.
            is_suppressed = pd.Series(False, index=chunk.index)
            if 'contains_override_data' in chunk:
                is_suppressed |= chunk['contains_override_data'].astype('string').str.strip().str.lower().isin(['true', '1', 'yes']).fillna(False)
            chunk['is_suppressed_override'] = is_suppressed
            chunk.loc[is_suppressed, ['customers_out', 'percent_out']] = np.nan

            chunk = chunk.dropna(subset=['time', 'county_fips'])
            chunk = chunk[(chunk['time'] >= RUN_START) & (chunk['time'] <= RUN_END)].copy()
            if not chunk.empty:
                chunk['time'] = chunk['time'].dt.floor('15min')
                yield chunk[['time', 'county_fips', 'county_name', 'state_abbr', 'customers_out', 'total_customers', 'percent_out', 'is_suppressed_override']]


def build_eaglei_county_panel(paths: Iterable[Path]) -> pd.DataFrame:
    pieces = []
    for chunk in read_eaglei_panel_chunks(paths):
        pieces.append(
            chunk.groupby(['time', 'county_fips', 'county_name', 'state_abbr'], dropna=False, as_index=False)
            .agg(
                customers_out=('customers_out', 'max'),
                total_customers=('total_customers', 'max'),
                percent_out=('percent_out', 'max'),
                is_suppressed_override=('is_suppressed_override', 'max'),
            )
        )
    if not pieces:
        return pd.DataFrame(columns=['time', 'county_fips', 'county_name', 'state_abbr', 'customers_out', 'total_customers', 'percent_out', 'is_suppressed_override'])
    panel = pd.concat(pieces, ignore_index=True)
    panel = (
        panel.groupby(['time', 'county_fips'], dropna=False, as_index=False)
        .agg(
            county_name=('county_name', 'first'),
            state_abbr=('state_abbr', 'first'),
            customers_out=('customers_out', 'max'),
            total_customers=('total_customers', 'max'),
            percent_out=('percent_out', 'max'),
            is_suppressed_override=('is_suppressed_override', 'max'),
        )
        .sort_values(['county_fips', 'time'])
        .reset_index(drop=True)
    )
    return panel


def county_episode_thresholds(panel: pd.DataFrame) -> pd.DataFrame:
    thresholds = (
        panel.groupby('county_fips', as_index=False)
        .agg(
            county_name=('county_name', 'first'),
            state_abbr=('state_abbr', 'first'),
            p99_customers_out=('customers_out', lambda values: values.quantile(0.99)),
            p99_percent_out=('percent_out', lambda values: values.quantile(0.99)),
        )
    )
    thresholds['customers_out_threshold'] = np.maximum(thresholds['p99_customers_out'], OUTAGE_EPISODE_MIN_CUSTOMERS)
    thresholds['percent_out_threshold'] = np.maximum(thresholds['p99_percent_out'].fillna(0.0), OUTAGE_EPISODE_MIN_PERCENT_OUT)
    return thresholds


def _max_consecutive_run_with_bridging(active_times: pd.Series, suppressed_times: set, step: pd.Timedelta = EAGLEI_PANEL_CADENCE) -> int:
    """Length of the longest run of active readings, bridging gaps fully explained by suppressed rows.

    Two active readings are treated as consecutive if they are exactly one cadence step apart, or if
    every intervening cadence slot between them corresponds to a panel row that was suppressed as
    override/non-collected data (i.e. Eagle-I reported *something* there, we just do not trust the
    value). A gap is NOT bridged if any intervening slot is simply missing from the panel, or if it
    holds a genuine reading that fell back below threshold -- those are real breaks in the outage.
    """
    times_sorted = sorted(active_times.tolist())
    if not times_sorted:
        return 0
    max_run = 1
    current_run = 1
    for prev_time, curr_time in zip(times_sorted[:-1], times_sorted[1:]):
        gap = curr_time - prev_time
        if gap <= pd.Timedelta(0):
            continue
        n_steps = int(round(gap / step))
        bridged = False
        if n_steps == 1:
            bridged = True
        elif n_steps > 1:
            bridged = all((prev_time + k * step) in suppressed_times for k in range(1, n_steps))
        if bridged:
            current_run += 1
        else:
            current_run = 1
        max_run = max(max_run, current_run)
    return max_run


def identify_eaglei_impact_episodes(panel: pd.DataFrame, thresholds: pd.DataFrame) -> pd.DataFrame:
    if panel.empty:
        return pd.DataFrame()
    threshold_lookup = thresholds.set_index('county_fips')
    has_suppression_column = 'is_suppressed_override' in panel.columns
    rows = []
    n_dropped_for_persistence = 0
    for county_fips, county_rows in panel.groupby('county_fips', sort=False):
        if county_fips not in threshold_lookup.index:
            continue
        threshold = threshold_lookup.loc[county_fips]
        if has_suppression_column:
            suppressed_times = set(county_rows.loc[county_rows['is_suppressed_override'].fillna(False), 'time'])
        else:
            suppressed_times = set()
        active = county_rows[
            (county_rows['customers_out'] >= threshold['customers_out_threshold'])
            | (county_rows['percent_out'] >= threshold['percent_out_threshold'])
        ].sort_values('time')
        if active.empty:
            continue
        groups = active['time'].diff().gt(OUTAGE_EPISODE_MAX_GAP).fillna(True).cumsum()
        for _, episode_rows in active.groupby(groups):
            start = episode_rows['time'].min()
            end = episode_rows['time'].max()
            duration = end - start + pd.Timedelta('15min')
            if duration < OUTAGE_EPISODE_MIN_DURATION:
                continue
            if _max_consecutive_run_with_bridging(episode_rows['time'], suppressed_times) < OUTAGE_EPISODE_MIN_CONSECUTIVE_SAMPLES:
                n_dropped_for_persistence += 1
                continue
            peak_index = episode_rows['customers_out'].idxmax()
            rows.append({
                'impact_episode_id': f'eaglei_{county_fips}_{start:%Y%m%d%H%M}',
                'county_fips': county_fips,
                'county_name': episode_rows['county_name'].dropna().iloc[0] if episode_rows['county_name'].notna().any() else pd.NA,
                'state_abbr': episode_rows['state_abbr'].dropna().iloc[0] if episode_rows['state_abbr'].notna().any() else pd.NA,
                'start_time': start,
                'end_time': end,
                'duration_hours': duration / pd.Timedelta(hours=1),
                'peak_time': episode_rows.loc[peak_index, 'time'],
                'peak_customers_out': float(episode_rows.loc[peak_index, 'customers_out']),
                'peak_percent_out': float(episode_rows['percent_out'].max()),
                'integrated_customer_outage_hours': float(episode_rows['customers_out'].sum() * 0.25),
                'n_samples': int(len(episode_rows)),
                'customers_out_threshold': float(threshold['customers_out_threshold']),
                'percent_out_threshold': float(threshold['percent_out_threshold']),
            })
    print(f'Episodes dropped for failing persistence requirement (< {OUTAGE_EPISODE_MIN_CONSECUTIVE_SAMPLES} consecutive 15-min readings, bridging suppressed-override gaps): {n_dropped_for_persistence}')
    return pd.DataFrame(rows).sort_values(['start_time', 'county_fips']).reset_index(drop=True)


def summarize_impact_episodes(impact_episodes: pd.DataFrame, county_outage_panel: pd.DataFrame, top_n: int = 20):
    """Full-period QA summary and outlier check for impact episodes.

    Returns (qa, top_by_peak_customers_out, top_by_integrated_customer_outage_hours, outliers).

    Outlier check flags episodes that are physically impossible given the source data:
    - peak_percent_out > 100 (percent of customers out cannot exceed 100%)
    - peak_customers_out greater than the county's own observed total_customers ceiling
      in the panel (customers out cannot exceed the county's customer base)
    These catch upstream aggregation/units bugs (e.g. a duplicate-report max-aggregation
    artifact) before they distort top-N rankings or quiet-period matching. With the
    state-name normalization and override-data suppression now applied upstream in
    build_eaglei_county_panel, this should ideally flag zero episodes on a fresh run.
    """
    empty = pd.DataFrame()
    if impact_episodes.empty:
        return empty, empty, empty, empty

    duration = impact_episodes['duration_hours']
    if not county_outage_panel.empty and 'total_customers' in county_outage_panel:
        county_max_total_customers = county_outage_panel.groupby('county_fips')['total_customers'].max()
    else:
        county_max_total_customers = pd.Series(dtype=float)
    episode_county_cap = impact_episodes['county_fips'].map(county_max_total_customers)
    percent_out_over_100 = impact_episodes['peak_percent_out'] > 100.0
    customers_out_over_cap = episode_county_cap.notna() & (impact_episodes['peak_customers_out'] > episode_county_cap)

    n_suppressed_panel_rows = int(county_outage_panel['is_suppressed_override'].sum()) if 'is_suppressed_override' in county_outage_panel else 0

    qa = pd.DataFrame([
        {'metric': 'panel_rows', 'value': len(county_outage_panel)},
        {'metric': 'panel_counties', 'value': county_outage_panel['county_fips'].nunique() if not county_outage_panel.empty else 0},
        {'metric': 'panel_rows_suppressed_override', 'value': n_suppressed_panel_rows},
        {'metric': 'impact_episodes', 'value': len(impact_episodes)},
        {'metric': 'episode_counties', 'value': impact_episodes['county_fips'].nunique()},
        {'metric': 'median_duration_hours', 'value': float(duration.median())},
        {'metric': 'p25_duration_hours', 'value': float(duration.quantile(0.25))},
        {'metric': 'p75_duration_hours', 'value': float(duration.quantile(0.75))},
        {'metric': 'p90_duration_hours', 'value': float(duration.quantile(0.90))},
        {'metric': 'p99_duration_hours', 'value': float(duration.quantile(0.99))},
        {'metric': 'fraction_duration_le_1h', 'value': float((duration <= 1.0).mean())},
        {'metric': 'max_peak_customers_out', 'value': float(impact_episodes['peak_customers_out'].max())},
        {'metric': 'max_integrated_customer_outage_hours', 'value': float(impact_episodes['integrated_customer_outage_hours'].max())},
        {'metric': 'n_episodes_percent_out_gt_100', 'value': int(percent_out_over_100.sum())},
        {'metric': 'n_episodes_customers_out_gt_county_total', 'value': int(customers_out_over_cap.sum())},
    ])

    top_by_peak_customers_out = impact_episodes.sort_values('peak_customers_out', ascending=False).head(top_n)
    top_by_integrated_hours = impact_episodes.sort_values('integrated_customer_outage_hours', ascending=False).head(top_n)

    outlier_mask = percent_out_over_100 | customers_out_over_cap
    outliers = impact_episodes[outlier_mask].copy()
    outliers['flag_percent_out_gt_100'] = percent_out_over_100[outlier_mask]
    outliers['flag_customers_out_gt_county_total'] = customers_out_over_cap[outlier_mask]

    return qa, top_by_peak_customers_out, top_by_integrated_hours, outliers


if RUN_EAGLEI_PANEL_BUILD:
    eaglei_impact_files = eagle_i_csv_files_for_impact_window(RUN_START.year, RUN_END.year)
    print(f'Eagle-I files to scan: {len(eaglei_impact_files)}')
    county_outage_panel = build_eaglei_county_panel(eaglei_impact_files)
    county_outage_panel.to_csv(EAGLEI_COUNTY_PANEL_CSV, index=False)
    print(f'Wrote Eagle-I county panel: {EAGLEI_COUNTY_PANEL_CSV} ({len(county_outage_panel):,} rows)')
else:
    county_outage_panel = pd.read_csv(EAGLEI_COUNTY_PANEL_CSV, parse_dates=['time'], dtype={'county_fips': str}) if EAGLEI_COUNTY_PANEL_CSV.exists() else pd.DataFrame()
    print('Episode-builder functions defined. Set RUN_EAGLEI_PANEL_BUILD=True to scan Eagle-I files and rebuild the panel.')
    print(f'Cached county panel loaded: {len(county_outage_panel):,} rows')

print('Impact-episode identification runs in the threshold-builder cell below, once county thresholds are available.')


In [ ]:
# Full-period Eagle-I county threshold table builder, then impact-episode identification
#
# This cell estimates county-specific outage thresholds over the full 2024-2025 Eagle-I record,
# then identifies impact episodes using those thresholds (moved here from the episode-builder
# cell above on 2026-09-21, since episode identification needs the threshold table this cell
# produces -- doing it here lets both cells run top-to-bottom on a fully un-cached run without
# a file-not-found error or a redundant second full-file scan). Run this before using the
# May 10-12 pilot for scientific interpretation.

RUN_EAGLEI_THRESHOLD_BUILD = True
EAGLEI_COUNTY_THRESHOLDS_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_thresholds_2024_2025.csv'
EAGLEI_COUNTY_THRESHOLDS_QA_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_thresholds_qa_2024_2025.csv'

EAGLEI_THRESHOLD_QUANTILES = [0.95, 0.99, 0.995]
EAGLEI_THRESHOLD_MIN_CUSTOMERS = OUTAGE_EPISODE_MIN_CUSTOMERS
EAGLEI_THRESHOLD_MIN_PERCENT_OUT = OUTAGE_EPISODE_MIN_PERCENT_OUT


def exact_quantiles(values: pd.Series, quantiles: list[float]) -> pd.Series:
    numeric = pd.to_numeric(values, errors='coerce').dropna()
    if numeric.empty:
        return pd.Series({f'q{int(q * 1000):03d}': np.nan for q in quantiles})
    return pd.Series({f'q{int(q * 1000):03d}': float(numeric.quantile(q)) for q in quantiles})


def build_county_threshold_table_from_panel(panel: pd.DataFrame) -> pd.DataFrame:
    if panel.empty:
        return pd.DataFrame()
    customer_quantiles = (
        panel.groupby('county_fips')['customers_out']
        .apply(lambda values: exact_quantiles(values, EAGLEI_THRESHOLD_QUANTILES))
        .unstack()
        .add_prefix('customers_out_')
        .reset_index()
    )
    percent_quantiles = (
        panel.groupby('county_fips')['percent_out']
        .apply(lambda values: exact_quantiles(values, EAGLEI_THRESHOLD_QUANTILES))
        .unstack()
        .add_prefix('percent_out_')
        .reset_index()
    )
    metadata = (
        panel.groupby('county_fips', as_index=False)
        .agg(
            county_name=('county_name', 'first'),
            state_abbr=('state_abbr', 'first'),
            n_samples=('customers_out', 'size'),
            max_customers_out=('customers_out', 'max'),
            max_percent_out=('percent_out', 'max'),
        )
    )
    out = metadata.merge(customer_quantiles, on='county_fips', how='left').merge(percent_quantiles, on='county_fips', how='left')
    out['customers_out_threshold'] = np.maximum(out['customers_out_q990'], EAGLEI_THRESHOLD_MIN_CUSTOMERS)
    out['percent_out_threshold'] = np.maximum(out['percent_out_q990'].fillna(0.0), EAGLEI_THRESHOLD_MIN_PERCENT_OUT)
    out['severe_customers_out_threshold'] = np.maximum(out['customers_out_q995'], out['customers_out_threshold'])
    out['severe_percent_out_threshold'] = np.maximum(out['percent_out_q995'].fillna(0.0), out['percent_out_threshold'])
    out['county_fips'] = out['county_fips'].astype(str).str.zfill(5)
    return out.sort_values('county_fips').reset_index(drop=True)


def load_or_build_county_panel_for_thresholds() -> pd.DataFrame:
    # Reuse the panel built in the episode-builder cell above if this is the same kernel
    # session -- avoids a second full Eagle-I file scan on an un-cached run.
    if 'county_outage_panel' in globals() and isinstance(county_outage_panel, pd.DataFrame) and not county_outage_panel.empty:
        print(f'Reusing county panel already built earlier in this session ({len(county_outage_panel):,} rows) -- no re-scan needed.')
        return county_outage_panel
    if EAGLEI_COUNTY_PANEL_CSV.exists():
        print(f'Loading cached county panel: {EAGLEI_COUNTY_PANEL_CSV}')
        return pd.read_csv(EAGLEI_COUNTY_PANEL_CSV, parse_dates=['time'], dtype={'county_fips': str})
    files = eagle_i_csv_files_for_impact_window(RUN_START.year, RUN_END.year)
    print(f'Building county panel from Eagle-I files: {len(files)}')
    panel = build_eaglei_county_panel(files)
    panel.to_csv(EAGLEI_COUNTY_PANEL_CSV, index=False)
    return panel


if RUN_EAGLEI_THRESHOLD_BUILD:
    county_outage_panel = load_or_build_county_panel_for_thresholds()
    county_thresholds = build_county_threshold_table_from_panel(county_outage_panel)
    county_thresholds.to_csv(EAGLEI_COUNTY_THRESHOLDS_CSV, index=False)
    threshold_qa = pd.DataFrame([
        {'metric': 'panel_rows', 'value': int(len(county_outage_panel))},
        {'metric': 'panel_counties', 'value': int(county_outage_panel['county_fips'].nunique()) if not county_outage_panel.empty else 0},
        {'metric': 'threshold_counties', 'value': int(county_thresholds['county_fips'].nunique()) if not county_thresholds.empty else 0},
        {'metric': 'median_customers_out_threshold', 'value': float(county_thresholds['customers_out_threshold'].median()) if not county_thresholds.empty else np.nan},
        {'metric': 'median_percent_out_threshold', 'value': float(county_thresholds['percent_out_threshold'].median()) if not county_thresholds.empty else np.nan},
        {'metric': 'max_customers_out_threshold', 'value': float(county_thresholds['customers_out_threshold'].max()) if not county_thresholds.empty else np.nan},
        {'metric': 'max_percent_out_threshold', 'value': float(county_thresholds['percent_out_threshold'].max()) if not county_thresholds.empty else np.nan},
    ])
    threshold_qa.to_csv(EAGLEI_COUNTY_THRESHOLDS_QA_CSV, index=False)
    print(f'Wrote threshold table: {EAGLEI_COUNTY_THRESHOLDS_CSV}')
    print(f'Wrote threshold QA: {EAGLEI_COUNTY_THRESHOLDS_QA_CSV}')
    display(threshold_qa)
    display(county_thresholds.sort_values('customers_out_threshold', ascending=False).head(20))

    # Impact-episode identification lives here, not in the episode-builder cell above, because it
    # needs county_thresholds, which this cell just built. county_outage_panel is already in
    # memory (built here or reused from the cell above), so this does not rescan raw files.
    impact_episodes = identify_eaglei_impact_episodes(county_outage_panel, county_thresholds)
    impact_episodes.to_csv(EAGLEI_IMPACT_EPISODES_CSV, index=False)
    print(f'Wrote Eagle-I impact episodes: {EAGLEI_IMPACT_EPISODES_CSV} ({len(impact_episodes):,} episodes)')
else:
    county_thresholds = pd.read_csv(EAGLEI_COUNTY_THRESHOLDS_CSV, dtype={'county_fips': str}) if EAGLEI_COUNTY_THRESHOLDS_CSV.exists() else pd.DataFrame()
    impact_episodes = pd.read_csv(EAGLEI_IMPACT_EPISODES_CSV, parse_dates=['start_time', 'end_time', 'peak_time'], dtype={'county_fips': str}) if EAGLEI_IMPACT_EPISODES_CSV.exists() else pd.DataFrame()
    print('Full-period threshold builder defined. Set RUN_EAGLEI_THRESHOLD_BUILD=True to scan Eagle-I files, build thresholds, and identify impact episodes.')
    print(f'Cached threshold counties loaded: {county_thresholds["county_fips"].nunique() if not county_thresholds.empty else 0:,}')
    print(f'Cached impact episodes loaded: {len(impact_episodes):,}')

episode_qa, episode_top_peak, episode_top_integrated, episode_outliers = summarize_impact_episodes(
    impact_episodes, county_outage_panel
)
if not episode_qa.empty:
    episode_qa.to_csv(EAGLEI_EPISODE_QA_CSV, index=False)
if not episode_outliers.empty:
    episode_outliers.to_csv(EAGLEI_EPISODE_OUTLIERS_CSV, index=False)

print('--- Full-period impact episode QA ---')
display(episode_qa)

print(f"Outlier episodes flagged (peak_percent_out > 100% or peak_customers_out > county's observed total_customers): {len(episode_outliers)}")
if not episode_outliers.empty:
    display(
        episode_outliers[
            ['impact_episode_id', 'county_fips', 'county_name', 'start_time', 'end_time',
             'peak_customers_out', 'peak_percent_out',
             'flag_percent_out_gt_100', 'flag_customers_out_gt_county_total']
        ].head(20)
    )

print('--- Top 20 episodes by peak_customers_out ---')
display(episode_top_peak)

print('--- Top 20 episodes by integrated_customer_outage_hours ---')
display(episode_top_integrated)

In [ ]:
# Pilot Eagle-I impact episode build for the May 2024 Gannon storm window
#
# This pilot is intentionally small. It scans only Eagle-I CSV files whose names include dates
# from 2024-05-10 through 2024-05-12, then writes pilot outputs under EAGLEI_PROCESSED_DIR.
# (Moved off OUTPUT_DIR on 2026-09-21: OUTPUT_DIR/space_weather/processed_data is reserved for
# results from aligning Eagle-I impact episodes against GeoE data, not Eagle-I-only outputs.)

RUN_EAGLEI_PILOT_EPISODE_BUILD = True
PILOT_START = pd.Timestamp('2024-05-10')
PILOT_END = pd.Timestamp('2024-05-12 23:59:59')
PILOT_LABEL = f'{PILOT_START:%Y%m%d}_{PILOT_END:%Y%m%d}'
PILOT_COUNTY_PANEL_CSV = EAGLEI_PROCESSED_DIR / f'eaglei_county_15min_panel_pilot_{PILOT_LABEL}.csv'
PILOT_IMPACT_EPISODES_CSV = EAGLEI_PROCESSED_DIR / f'eaglei_impact_episodes_pilot_{PILOT_LABEL}.csv'
PILOT_EPISODE_QA_CSV = EAGLEI_PROCESSED_DIR / f'eaglei_impact_episode_qa_pilot_{PILOT_LABEL}.csv'


def eagle_i_csv_files_for_date_window(start_time: pd.Timestamp, end_time: pd.Timestamp) -> list[Path]:
    wanted_dates = {f'{day:%Y%m%d}' for day in pd.date_range(start_time.normalize(), end_time.normalize(), freq='D')}
    files = []
    for path in sorted(EAGLEI_DIR.rglob('*.csv')):
        if EAGLEI_PROCESSED_DIR in path.parents:
            continue
        if any(date_text in path.name for date_text in wanted_dates):
            files.append(path)
    return files


def subset_panel_to_window(panel: pd.DataFrame, start_time: pd.Timestamp, end_time: pd.Timestamp) -> pd.DataFrame:
    if panel.empty:
        return panel.copy()
    return panel[(panel['time'] >= start_time) & (panel['time'] <= end_time)].copy()


if RUN_EAGLEI_PILOT_EPISODE_BUILD:
    pilot_files = eagle_i_csv_files_for_date_window(PILOT_START, PILOT_END)
    print(f'Pilot Eagle-I files: {len(pilot_files)}')
    pilot_county_panel = subset_panel_to_window(build_eaglei_county_panel(pilot_files), PILOT_START, PILOT_END)
    pilot_county_panel.to_csv(PILOT_COUNTY_PANEL_CSV, index=False)

    # pilot_thresholds = county_episode_thresholds(pilot_county_panel)
    pilot_thresholds = pd.read_csv(EAGLEI_COUNTY_THRESHOLDS_CSV, dtype={'county_fips': str})
    pilot_impact_episodes = identify_eaglei_impact_episodes(pilot_county_panel, pilot_thresholds)
    pilot_impact_episodes.to_csv(PILOT_IMPACT_EPISODES_CSV, index=False)

    pilot_qa = pd.DataFrame([
        {'metric': 'pilot_files', 'value': len(pilot_files)},
        {'metric': 'panel_rows', 'value': len(pilot_county_panel)},
        {'metric': 'panel_counties', 'value': pilot_county_panel['county_fips'].nunique() if not pilot_county_panel.empty else 0},
        {'metric': 'impact_episodes', 'value': len(pilot_impact_episodes)},
        {'metric': 'episode_counties', 'value': pilot_impact_episodes['county_fips'].nunique() if not pilot_impact_episodes.empty else 0},
        {'metric': 'median_duration_hours', 'value': float(pilot_impact_episodes['duration_hours'].median()) if not pilot_impact_episodes.empty else np.nan},
        {'metric': 'max_peak_customers_out', 'value': float(pilot_impact_episodes['peak_customers_out'].max()) if not pilot_impact_episodes.empty else np.nan},
    ])
    pilot_qa.to_csv(PILOT_EPISODE_QA_CSV, index=False)
    print(f'Wrote pilot county panel: {PILOT_COUNTY_PANEL_CSV}')
    print(f'Wrote pilot impact episodes: {PILOT_IMPACT_EPISODES_CSV}')
    display(pilot_qa)
    if not pilot_impact_episodes.empty:
        display(pilot_impact_episodes.sort_values('peak_customers_out', ascending=False).head(20))
else:
    pilot_county_panel = pd.read_csv(PILOT_COUNTY_PANEL_CSV, parse_dates=['time'], dtype={'county_fips': str}) if PILOT_COUNTY_PANEL_CSV.exists() else pd.DataFrame()
    pilot_impact_episodes = pd.read_csv(PILOT_IMPACT_EPISODES_CSV, parse_dates=['start_time', 'end_time', 'peak_time'], dtype={'county_fips': str}) if PILOT_IMPACT_EPISODES_CSV.exists() else pd.DataFrame()
    print('Pilot episode builder defined. Set RUN_EAGLEI_PILOT_EPISODE_BUILD=True to run the May 10-12 pilot.')
    print(f'Cached pilot episodes loaded: {len(pilot_impact_episodes):,}')

In [ ]:
# Reset expensive execution flags after cache-building runs
#
# Keep these false during normal notebook navigation. Flip individual flags only for the
# specific cell you intend to run.

RUN_EAGLEI_THRESHOLD_BUILD = False
RUN_EAGLEI_PILOT_EPISODE_BUILD = False
RUN_EAGLEI_PANEL_BUILD = False
RUN_EAGLEI_QUIET_MATCHING = False
RUN_GEOE_CACHE_BUILD = False

print('Expensive run flags reset to False.')

In [23]:
# Local time-decoding helper used by the copied GeoE interpolation code


def _decode_time_coord_from_dataarray(da: xr.DataArray, fallback_base_date: str) -> pd.DatetimeIndex:
    if 'time' not in da.dims:
        return pd.DatetimeIndex([])

    t_da = da['time']
    if np.issubdtype(t_da.dtype, np.datetime64):
        return pd.DatetimeIndex(pd.to_datetime(t_da.values, errors='coerce'))

    vals = np.asarray(t_da.values, dtype=float)
    vals[~np.isfinite(vals)] = np.nan
    vals[np.abs(vals) > 1e20] = np.nan

    attrs_lc = {str(key).lower(): value for key, value in t_da.attrs.items()}
    units_attr = str(attrs_lc.get('units', '')).lower()
    reftime_attr = attrs_lc.get('reftime', None)

    if 'since' in units_attr:
        unit_map = {
            'seconds': 's', 'second': 's', 'sec': 's',
            'minutes': 'm', 'minute': 'm', 'min': 'm',
            'hours': 'h', 'hour': 'h', 'hr': 'h',
            'days': 'D', 'day': 'D',
            'milliseconds': 'ms', 'millisecond': 'ms', 'msec': 'ms',
            'microseconds': 'us', 'microsecond': 'us', 'usec': 'us',
            'nanoseconds': 'ns', 'nanosecond': 'ns', 'nsec': 'ns',
        }
        parsed_unit = next((value for key, value in unit_map.items() if key in units_attr), None)
        if parsed_unit is not None:
            try:
                origin = pd.Timestamp(units_attr.split('since', 1)[1].strip())
                return pd.DatetimeIndex(pd.to_datetime(vals, unit=parsed_unit, origin=origin, errors='coerce'))
            except Exception:
                pass

    if reftime_attr is not None:
        try:
            return pd.DatetimeIndex(pd.Timestamp(str(reftime_attr)) + pd.to_timedelta(vals, unit='s'))
        except Exception:
            pass

    finite_vals = vals[np.isfinite(vals)]
    if finite_vals.size > 0 and np.nanmin(finite_vals) >= 0 and np.nanmax(finite_vals) <= 172800:
        base = pd.Timestamp(fallback_base_date).normalize()
        return pd.DatetimeIndex(base + pd.to_timedelta(vals, unit='s'))

    return pd.DatetimeIndex(pd.to_datetime(vals, errors='coerce'))


print('GeoE time-decoding helper ready.')

GeoE time-decoding helper ready.


In [24]:
# Space Weather Interpolation

# GeoE processing fixes: fill-value masking and interpolation-based regridding
#
# This overrides `_grid_space_weather_to_analysis_grid` from the utility cell above. The NOAA geoE
# product is modeled on an unstructured point grid; assigning those points to nearest 50 km cells
# leaves empty analysis cells and visible striping. Interpolate Ex/Ey from the modeled point field
# to the analysis grid, and mask IDL/NetCDF fill values before magnitude or thresholds are computed.

from typing import Optional, Tuple

SPACE_WEATHER_FILL_ABS_THRESHOLD = 1e20


def _space_weather_fill_value_candidates(da: xr.DataArray) -> list:
    candidates = []
    for mapping in (getattr(da, "attrs", {}), getattr(da, "encoding", {})):
        for key in ("_FillValue", "missing_value"):
            if key not in mapping:
                continue
            value = mapping[key]
            values = np.ravel(value) if np.ndim(value) else [value]
            for item in values:
                try:
                    number = float(item)
                except (TypeError, ValueError):
                    continue
                if np.isfinite(number):
                    candidates.append(number)
    return candidates


def _mask_space_weather_fill_values(values: np.ndarray, da: Optional[xr.DataArray] = None) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32).copy()
    invalid = ~np.isfinite(arr) | (np.abs(arr) > SPACE_WEATHER_FILL_ABS_THRESHOLD)
    if da is not None:
        for fill_value in _space_weather_fill_value_candidates(da):
            invalid |= arr == np.float32(fill_value)
    arr[invalid] = np.nan
    return arr


def _space_weather_site_coordinates(ds_space: xr.Dataset) -> Tuple[str, np.ndarray, np.ndarray, np.ndarray]:
    lat_name = "latitude" if ("latitude" in ds_space.coords or "latitude" in ds_space.data_vars) else "lat"
    lon_name = "longitude" if ("longitude" in ds_space.coords or "longitude" in ds_space.data_vars) else "lon"

    if lat_name in ds_space.data_vars:
        ds_space = ds_space.assign_coords({lat_name: ds_space[lat_name]})
    if lon_name in ds_space.data_vars:
        ds_space = ds_space.assign_coords({lon_name: ds_space[lon_name]})

    if lat_name not in ds_space.coords or lon_name not in ds_space.coords:
        raise ValueError("Space-weather dataset must include latitude/longitude as coords or data variables.")

    lat_da = ds_space[lat_name]
    lon_da = ds_space[lon_name]

    if len(lat_da.dims) == 1 and len(lon_da.dims) == 1 and lat_da.dims[0] == lon_da.dims[0]:
        site_dim = lat_da.dims[0]
        src_lat = _mask_space_weather_fill_values(lat_da.values, lat_da).astype(float)
        src_lon = _mask_space_weather_fill_values(lon_da.values, lon_da).astype(float)
    else:
        if "time" not in lat_da.dims or "time" not in lon_da.dims:
            raise ValueError("Space-weather latitude/longitude must be 1D site or 2D (time,site).")
        other_lat = [dim for dim in lat_da.dims if dim != "time"]
        other_lon = [dim for dim in lon_da.dims if dim != "time"]
        if len(other_lat) != 1 or len(other_lon) != 1 or other_lat[0] != other_lon[0]:
            raise ValueError("Could not infer space-weather site dimension from latitude/longitude.")

        site_dim = other_lat[0]
        lat_2d = _mask_space_weather_fill_values(lat_da.transpose("time", site_dim).values, lat_da).astype(float)
        lon_2d = _mask_space_weather_fill_values(lon_da.transpose("time", site_dim).values, lon_da).astype(float)
        src_lat = np.nanmedian(lat_2d, axis=0)
        src_lon = np.nanmedian(lon_2d, axis=0)

        bad = ~np.isfinite(src_lat) | ~np.isfinite(src_lon)
        if np.any(bad):
            valid_matrix = np.isfinite(lat_2d) & np.isfinite(lon_2d)
            for site_index in np.where(bad)[0]:
                rows = np.where(valid_matrix[:, site_index])[0]
                if len(rows) > 0:
                    row_index = rows[0]
                    src_lat[site_index] = lat_2d[row_index, site_index]
                    src_lon[site_index] = lon_2d[row_index, site_index]

    keep_sites = np.isfinite(src_lat) & np.isfinite(src_lon)
    if not keep_sites.any():
        raise ValueError("No finite space-weather site coordinates were available for gridding.")

    return site_dim, src_lat[keep_sites], src_lon[keep_sites], keep_sites


def _build_space_weather_interpolation(src_lat: np.ndarray, src_lon: np.ndarray, ds_target_grid: xr.Dataset) -> dict:
    if "lat" not in ds_target_grid or "lon" not in ds_target_grid:
        raise ValueError("Target analysis grid must contain 2D lat/lon variables.")

    target_lat = ds_target_grid["lat"].values.astype(float).ravel()
    target_lon = ds_target_grid["lon"].values.astype(float).ravel()
    target_valid = np.isfinite(target_lat) & np.isfinite(target_lon)
    source_points = np.column_stack([src_lon, src_lat])
    target_points = np.column_stack([target_lon, target_lat])

    nearest_source = cKDTree(source_points).query(target_points, k=1)[1].astype(np.int64)
    simplex = np.full(target_points.shape[0], -1, dtype=np.int64)
    vertices = np.full((target_points.shape[0], 3), -1, dtype=np.int64)
    weights = np.full((target_points.shape[0], 3), np.nan, dtype=np.float32)

    if source_points.shape[0] >= 3:
        triangulation = Delaunay(source_points)
        valid_target_indices = np.where(target_valid)[0]
        simplex_valid = triangulation.find_simplex(target_points[valid_target_indices])
        inside_target_indices = valid_target_indices[simplex_valid >= 0]
        inside_simplex = simplex_valid[simplex_valid >= 0]
        if inside_target_indices.size:
            transform = triangulation.transform[inside_simplex]
            delta = target_points[inside_target_indices] - transform[:, 2]
            first_weights = np.einsum("ijk,ik->ij", transform[:, :2, :], delta)
            barycentric = np.column_stack([first_weights, 1.0 - first_weights.sum(axis=1)]).astype(np.float32)
            simplex[inside_target_indices] = inside_simplex
            vertices[inside_target_indices] = triangulation.simplices[inside_simplex]
            weights[inside_target_indices] = barycentric

    return {
        "target_valid": target_valid,
        "nearest_source": nearest_source,
        "inside": simplex >= 0,
        "vertices": vertices,
        "weights": weights,
        "shape": (int(ds_target_grid.sizes["y"]), int(ds_target_grid.sizes["x"])),
    }


def _interpolate_space_weather_rows(values: np.ndarray, interpolation: dict) -> np.ndarray:
    arr = _mask_space_weather_fill_values(values)
    nt = arr.shape[0]
    ncells = interpolation["target_valid"].size
    out = np.full((nt, ncells), np.nan, dtype=np.float32)

    inside = interpolation["inside"]
    inside_indices = np.where(inside)[0]
    vertices = interpolation["vertices"][inside_indices]
    weights = interpolation["weights"][inside_indices]
    nearest_source = interpolation["nearest_source"]
    target_valid = interpolation["target_valid"]

    for time_index in range(nt):
        row = arr[time_index]
        gridded = np.full(ncells, np.nan, dtype=np.float32)
        if np.isfinite(row).any():
            if inside_indices.size:
                vertex_values = row[vertices]
                valid_vertices = np.isfinite(vertex_values).all(axis=1)
                if np.any(valid_vertices):
                    gridded[inside_indices[valid_vertices]] = np.sum(
                        vertex_values[valid_vertices] * weights[valid_vertices], axis=1
                    ).astype(np.float32)

            fallback = np.isnan(gridded) & target_valid & np.isfinite(row[nearest_source])
            gridded[fallback] = row[nearest_source[fallback]]
        out[time_index] = gridded

    return out


def _grid_space_weather_to_analysis_grid(ds_space_raw: xr.Dataset, ds_target_grid: xr.Dataset) -> xr.Dataset:
    if ds_space_raw is None:
        raise ValueError("ds_space_raw is None")

    ds_space = ds_space_raw.copy()
    if "time" in ds_space.coords:
        fake_da = xr.DataArray(np.zeros(ds_space.sizes["time"]), dims=("time",), coords={"time": ds_space["time"]})
        decoded_time = _decode_time_coord_from_dataarray(fake_da, fallback_base_date=TIME_START)
        valid_time = ~pd.isna(decoded_time)
        if not np.all(valid_time):
            ds_space = ds_space.isel(time=np.where(valid_time)[0])
            decoded_time = decoded_time[valid_time]
        ds_space = ds_space.assign_coords(time=pd.DatetimeIndex(decoded_time))

    site_dim, src_lat, src_lon, keep_sites = _space_weather_site_coordinates(ds_space)
    interpolation = _build_space_weather_interpolation(src_lat, src_lon, ds_target_grid)
    ny, nx = interpolation["shape"]

    out_vars = {}
    for var_name, da in ds_space.data_vars.items():
        if "time" not in da.dims or site_dim not in da.dims:
            continue

        arr = da.transpose("time", site_dim).values
        if arr.shape[1] == keep_sites.size:
            arr = arr[:, keep_sites]
        arr = _mask_space_weather_fill_values(arr, da)
        gridded = _interpolate_space_weather_rows(arr, interpolation)
        out_vars[var_name] = (("time", "y", "x"), gridded.reshape(arr.shape[0], ny, nx))

    ds_out = xr.Dataset(
        data_vars=out_vars,
        coords={
            "time": ds_space["time"].values,
            "y": ds_target_grid["y"].values,
            "x": ds_target_grid["x"].values,
            "lat": (("y", "x"), ds_target_grid["lat"].values),
            "lon": (("y", "x"), ds_target_grid["lon"].values),
        },
        attrs={
            "gridding_method": "linear_delaunay_interpolation_with_nearest_fallback",
            "source_grid": "NOAA modeled geoE unstructured points",
            "fill_value_policy": f"masked nonfinite values and abs(value) > {SPACE_WEATHER_FILL_ABS_THRESHOLD:.1e}",
        },
    )
    ds_out["space_weather_linear_interpolation_support"] = (
        ("y", "x"), interpolation["inside"].reshape(ny, nx).astype(np.uint8)
    )
    ds_out["space_weather_nearest_source_index"] = (
        ("y", "x"), interpolation["nearest_source"].reshape(ny, nx).astype(np.int32)
    )
    return ds_out


print("GeoE mapper override ready: fill values masked; Ex/Ey interpolated to the analysis grid with nearest fallback.")

# GeoE no-extrapolation cutoff: unsupported cells become NaN
#
# Linear interpolation is retained inside the modeled geoE point cloud. Outside that support,
# nearest-source fallback is allowed only for cells within 100 km of the modeled points; farther
# cells are outside the current CONUS-supported analysis domain and remain NaN.

SPACE_WEATHER_MAX_FALLBACK_DISTANCE_KM = 100.0


def _approx_latlon_distance_km(lat1: np.ndarray, lon1: np.ndarray, lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    mean_lat_rad = np.deg2rad((lat1 + lat2) / 2.0)
    dy_km = (lat1 - lat2) * 111.32
    dx_km = (lon1 - lon2) * 111.32 * np.cos(mean_lat_rad)
    return np.hypot(dx_km, dy_km)


def _build_space_weather_interpolation(src_lat: np.ndarray, src_lon: np.ndarray, ds_target_grid: xr.Dataset) -> dict:
    if "lat" not in ds_target_grid or "lon" not in ds_target_grid:
        raise ValueError("Target analysis grid must contain 2D lat/lon variables.")

    target_lat = ds_target_grid["lat"].values.astype(float).ravel()
    target_lon = ds_target_grid["lon"].values.astype(float).ravel()
    target_valid = np.isfinite(target_lat) & np.isfinite(target_lon)
    source_points = np.column_stack([src_lon, src_lat])
    target_points = np.column_stack([target_lon, target_lat])

    nearest_source = cKDTree(source_points).query(target_points, k=1)[1].astype(np.int64)
    nearest_distance_km = _approx_latlon_distance_km(
        target_lat,
        target_lon,
        src_lat[nearest_source],
        src_lon[nearest_source],
    ).astype(np.float32)
    nearest_fallback_supported = target_valid & (nearest_distance_km < SPACE_WEATHER_MAX_FALLBACK_DISTANCE_KM)

    simplex = np.full(target_points.shape[0], -1, dtype=np.int64)
    vertices = np.full((target_points.shape[0], 3), -1, dtype=np.int64)
    weights = np.full((target_points.shape[0], 3), np.nan, dtype=np.float32)

    if source_points.shape[0] >= 3:
        triangulation = Delaunay(source_points)
        valid_target_indices = np.where(target_valid)[0]
        simplex_valid = triangulation.find_simplex(target_points[valid_target_indices])
        inside_target_indices = valid_target_indices[simplex_valid >= 0]
        inside_simplex = simplex_valid[simplex_valid >= 0]
        if inside_target_indices.size:
            transform = triangulation.transform[inside_simplex]
            delta = target_points[inside_target_indices] - transform[:, 2]
            first_weights = np.einsum("ijk,ik->ij", transform[:, :2, :], delta)
            barycentric = np.column_stack([first_weights, 1.0 - first_weights.sum(axis=1)]).astype(np.float32)
            simplex[inside_target_indices] = inside_simplex
            vertices[inside_target_indices] = triangulation.simplices[inside_simplex]
            weights[inside_target_indices] = barycentric

    return {
        "target_valid": target_valid,
        "nearest_source": nearest_source,
        "nearest_source_distance_km": nearest_distance_km,
        "nearest_fallback_supported": nearest_fallback_supported,
        "inside": simplex >= 0,
        "vertices": vertices,
        "weights": weights,
        "shape": (int(ds_target_grid.sizes["y"]), int(ds_target_grid.sizes["x"])),
    }


def _interpolate_space_weather_rows(values: np.ndarray, interpolation: dict) -> np.ndarray:
    arr = _mask_space_weather_fill_values(values)
    nt = arr.shape[0]
    ncells = interpolation["target_valid"].size
    out = np.full((nt, ncells), np.nan, dtype=np.float32)

    inside_indices = np.where(interpolation["inside"])[0]
    vertices = interpolation["vertices"][inside_indices]
    weights = interpolation["weights"][inside_indices]
    nearest_source = interpolation["nearest_source"]
    nearest_fallback_supported = interpolation["nearest_fallback_supported"]

    for time_index in range(nt):
        row = arr[time_index]
        gridded = np.full(ncells, np.nan, dtype=np.float32)
        if np.isfinite(row).any():
            if inside_indices.size:
                vertex_values = row[vertices]
                valid_vertices = np.isfinite(vertex_values).all(axis=1)
                if np.any(valid_vertices):
                    gridded[inside_indices[valid_vertices]] = np.sum(
                        vertex_values[valid_vertices] * weights[valid_vertices], axis=1
                    ).astype(np.float32)

            fallback = np.isnan(gridded) & nearest_fallback_supported & np.isfinite(row[nearest_source])
            gridded[fallback] = row[nearest_source[fallback]]
        out[time_index] = gridded

    return out


def _grid_space_weather_to_analysis_grid(ds_space_raw: xr.Dataset, ds_target_grid: xr.Dataset) -> xr.Dataset:
    if ds_space_raw is None:
        raise ValueError("ds_space_raw is None")

    ds_space = ds_space_raw.copy()
    if "time" in ds_space.coords:
        fake_da = xr.DataArray(np.zeros(ds_space.sizes["time"]), dims=("time",), coords={"time": ds_space["time"]})
        decoded_time = _decode_time_coord_from_dataarray(fake_da, fallback_base_date=TIME_START)
        valid_time = ~pd.isna(decoded_time)
        if not np.all(valid_time):
            ds_space = ds_space.isel(time=np.where(valid_time)[0])
            decoded_time = decoded_time[valid_time]
        ds_space = ds_space.assign_coords(time=pd.DatetimeIndex(decoded_time))

    site_dim, src_lat, src_lon, keep_sites = _space_weather_site_coordinates(ds_space)
    interpolation = _build_space_weather_interpolation(src_lat, src_lon, ds_target_grid)
    ny, nx = interpolation["shape"]

    out_vars = {}
    for var_name, da in ds_space.data_vars.items():
        if "time" not in da.dims or site_dim not in da.dims:
            continue

        arr = da.transpose("time", site_dim).values
        if arr.shape[1] == keep_sites.size:
            arr = arr[:, keep_sites]
        arr = _mask_space_weather_fill_values(arr, da)
        gridded = _interpolate_space_weather_rows(arr, interpolation)
        out_vars[var_name] = (("time", "y", "x"), gridded.reshape(arr.shape[0], ny, nx))

    ds_out = xr.Dataset(
        data_vars=out_vars,
        coords={
            "time": ds_space["time"].values,
            "y": ds_target_grid["y"].values,
            "x": ds_target_grid["x"].values,
            "lat": (("y", "x"), ds_target_grid["lat"].values),
            "lon": (("y", "x"), ds_target_grid["lon"].values),
        },
        attrs={
            "gridding_method": "linear_delaunay_interpolation_with_limited_nearest_fallback",
            "source_grid": "NOAA modeled geoE unstructured points",
            "fill_value_policy": f"masked nonfinite values and abs(value) > {SPACE_WEATHER_FILL_ABS_THRESHOLD:.1e}",
            "fallback_policy": f"nearest fallback only where nearest source point is < {SPACE_WEATHER_MAX_FALLBACK_DISTANCE_KM:.0f} km away; farther cells are NaN",
        },
    )
    ds_out["space_weather_linear_interpolation_support"] = (
        ("y", "x"), interpolation["inside"].reshape(ny, nx).astype(np.uint8)
    )
    ds_out["space_weather_nearest_fallback_support"] = (
        ("y", "x"), interpolation["nearest_fallback_supported"].reshape(ny, nx).astype(np.uint8)
    )
    ds_out["space_weather_nearest_source_distance_km"] = (
        ("y", "x"), interpolation["nearest_source_distance_km"].reshape(ny, nx).astype(np.float32)
    )
    ds_out["space_weather_nearest_source_index"] = (
        ("y", "x"), interpolation["nearest_source"].reshape(ny, nx).astype(np.int32)
    )
    return ds_out


print("GeoE fallback cutoff ready: cells >=100 km from modeled support remain NaN.")

GeoE mapper override ready: fill values masked; Ex/Ey interpolated to the analysis grid with nearest fallback.
GeoE fallback cutoff ready: cells >=100 km from modeled support remain NaN.


In [25]:
# Generate or load two-year interpolated GeoE cache
#
# This cell is intentionally off by default. Set RUN_GEOE_CACHE_BUILD=True when you are ready
# to generate monthly NetCDF files from the daily NOAA geoE source files.

RUN_GEOE_CACHE_BUILD = True
GEOE_CACHE_VERSION = 'v1_fillmasked_interp_noextrap'
GEOE_CACHE_PATTERN = f'geoe_interpolated_{GEOE_CACHE_VERSION}_*.nc'
GEOE_MONTHLY_TEMPLATE = f'geoe_interpolated_{GEOE_CACHE_VERSION}_{{year:04d}}_{{month:02d}}.nc'
GEOE_CACHE_INDEX_CSV = GEOE_OUTPUT_DIR / f'geoe_interpolated_{GEOE_CACHE_VERSION}_index.csv'


def list_geoe_daily_files_for_window(start_time: pd.Timestamp, end_time: pd.Timestamp) -> pd.DataFrame:
    rows = []
    for path in sorted(SPACE_WEATHER_DIR.glob('*-empirical-EMTF-2022.12-2022.12.nc')):
        date_text = path.name.split('-', 1)[0]
        try:
            timestamp = pd.Timestamp(date_text)
        except Exception:
            continue
        if start_time.normalize() <= timestamp <= end_time.normalize():
            rows.append({'date': timestamp.normalize(), 'path': path})
    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)


def _geoe_month_groups(files: pd.DataFrame) -> Iterable[tuple[tuple[int, int], pd.DataFrame]]:
    if files.empty:
        return []
    grouped = files.assign(year=files['date'].dt.year, month=files['date'].dt.month).groupby(['year', 'month'], sort=True)
    return grouped


def build_monthly_geoe_cache(month_files: pd.DataFrame, output_path: Path, ds_target_grid: xr.Dataset) -> dict:
    datasets = []
    source_dates = []
    for row in month_files.itertuples(index=False):
        with xr.open_dataset(row.path, decode_times=False) as ds_raw:
            ds_grid = _grid_space_weather_to_analysis_grid(ds_raw, ds_target_grid)
            ex = ds_grid['Ex']
            ey = ds_grid['Ey']
            e_mag = xr.apply_ufunc(np.hypot, ex, ey).astype(np.float32)
            out = xr.Dataset(
                data_vars={
                    'Ex': ex.astype(np.float32),
                    'Ey': ey.astype(np.float32),
                    'E_mag': e_mag,
                    'space_weather_linear_interpolation_support': ds_grid['space_weather_linear_interpolation_support'],
                    'space_weather_nearest_fallback_support': ds_grid['space_weather_nearest_fallback_support'],
                    'space_weather_nearest_source_distance_km': ds_grid['space_weather_nearest_source_distance_km'],
                    'space_weather_nearest_source_index': ds_grid['space_weather_nearest_source_index'],
                },
                coords=ds_grid.coords,
                attrs={
                    **ds_grid.attrs,
                    'cache_version': GEOE_CACHE_VERSION,
                    'source_file': str(row.path),
                    'source_date': f'{row.date:%Y-%m-%d}',
                },
            )
            datasets.append(out)
            source_dates.append(row.date)
    if not datasets:
        raise ValueError('No GeoE daily files were provided for this month.')

    month_ds = xr.concat(datasets, dim='time').sortby('time')
    month_ds.attrs.update({
        'cache_version': GEOE_CACHE_VERSION,
        'source_product': 'NOAA geoE empirical EMTF Ex/Ey daily files',
        'processing': 'fill-value masking; linear Delaunay interpolation to analysis grid; nearest fallback only within configured distance; unsupported cells NaN',
        'source_start_date': f'{min(source_dates):%Y-%m-%d}',
        'source_end_date': f'{max(source_dates):%Y-%m-%d}',
        'n_source_files': int(len(source_dates)),
    })
    encoding = {
        'Ex': {'zlib': True, 'complevel': 4, 'dtype': 'float32'},
        'Ey': {'zlib': True, 'complevel': 4, 'dtype': 'float32'},
        'E_mag': {'zlib': True, 'complevel': 4, 'dtype': 'float32'},
    }
    month_ds.to_netcdf(output_path, encoding=encoding)
    month_ds.close()
    for ds in datasets:
        ds.close()
    return {'path': output_path, 'n_source_files': len(source_dates), 'start_date': min(source_dates), 'end_date': max(source_dates)}


def build_or_load_geoe_cache(run_build: bool = RUN_GEOE_CACHE_BUILD) -> xr.Dataset | None:
    files = list_geoe_daily_files_for_window(RUN_START, RUN_END)
    existing = sorted(GEOE_OUTPUT_DIR.glob(GEOE_CACHE_PATTERN))
    print(f'GeoE source daily files: {len(files)}')
    print(f'Existing monthly cache files: {len(existing)}')
    print(f'GeoE cache directory: {GEOE_OUTPUT_DIR}')

    if run_build:
        if files.empty:
            raise FileNotFoundError(f'No GeoE source files found under {SPACE_WEATHER_DIR}')
        rows = []
        with xr.open_dataset(GRID_PATH) as ds_target_grid:
            for (year, month), month_files in _geoe_month_groups(files):
                output_path = GEOE_OUTPUT_DIR / GEOE_MONTHLY_TEMPLATE.format(year=int(year), month=int(month))
                if output_path.exists():
                    print(f'Skipping existing cache: {output_path.name}')
                    rows.append({'year': int(year), 'month': int(month), 'path': str(output_path), 'status': 'exists', 'n_source_files': len(month_files)})
                    continue
                print(f'Building {output_path.name} from {len(month_files)} daily files')
                result = build_monthly_geoe_cache(month_files, output_path, ds_target_grid)
                rows.append({'year': int(year), 'month': int(month), 'path': str(result['path']), 'status': 'built', 'n_source_files': result['n_source_files']})
        pd.DataFrame(rows).to_csv(GEOE_CACHE_INDEX_CSV, index=False)
        existing = sorted(GEOE_OUTPUT_DIR.glob(GEOE_CACHE_PATTERN))
        print(f'Cache files after build: {len(existing)}')

    if not existing:
        print('No GeoE cache files loaded. Set RUN_GEOE_CACHE_BUILD=True to generate them.')
        return None

    ds = xr.open_mfdataset(existing, combine='by_coords', chunks={'time': 288})
    print(f'Loaded GeoE cache lazily: {len(existing)} files, time={ds.sizes.get("time", 0):,}, y={ds.sizes.get("y", 0)}, x={ds.sizes.get("x", 0)}')
    return ds


geoe_cache = build_or_load_geoe_cache()


GeoE source daily files: 727
Existing monthly cache files: 0
GeoE cache directory: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/processed_data
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_01.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_02.nc from 29 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_03.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_04.nc from 30 daily files


/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_05.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_06.nc from 30 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_07.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_08.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_09.nc from 30 daily files


/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_10.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_11.nc from 30 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2024_12.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_01.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_02.nc from 24 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_03.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_04.nc from 30 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_05.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_06.nc from 30 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_07.nc from 31 daily files
Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_08.nc from 31 daily files


...

Building geoe_interpolated_v1_fillmasked_interp_noextrap_2025_12.nc from 31 daily files


/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


Cache files after build: 24


/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 288. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 288. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 288. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
/Users/ryanmc/miniforge3/envs/spwxr_network_new/lib/python3.9/site-packages/xarray/core/

Loaded GeoE cache lazily: 24 files, time=1,040,793, y=55, x=104


## Matched Quiet Periods

Quiet periods should resemble each impact episode in background conditions but should not be selected to resemble the geoE hazard. Start with time-structure matching, then add weather/load/antecedent outage covariates as those fields are available. Because Eagle-I timing may include reporting latency, compare geoE descriptors over windows around each impact episode rather than only at the nominal timestamp.

In [ ]:
# Matched quiet periods for Eagle-I impact episodes
#
# This uses time-structure matching only for the first pass: same county, month, day-of-week,
# hour, and approximate episode duration. Later passes can add weather/load/antecedent outage
# matching once those covariates are assembled.

# Moved off OUTPUT_DIR on 2026-09-21: OUTPUT_DIR/space_weather/processed_data is reserved for
# results from aligning Eagle-I impact episodes against GeoE data, not Eagle-I-only outputs.
EAGLEI_QUIET_PERIODS_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_matched_quiet_periods_2024_2025.csv'
EAGLEI_QUIET_PERIOD_QA_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_matched_quiet_period_qa_2024_2025.csv'
RUN_EAGLEI_QUIET_MATCHING = False

QUIET_MATCHES_PER_EPISODE = 20
QUIET_EXCLUSION_BUFFER = pd.Timedelta('2D')
QUIET_RANDOM_SEED = 42


def construct_time_matched_quiet_periods(
    impact_episodes: pd.DataFrame,
    county_panel: pd.DataFrame,
    n_matches: int = QUIET_MATCHES_PER_EPISODE,
    exclusion_buffer: pd.Timedelta = QUIET_EXCLUSION_BUFFER,
    random_seed: int = QUIET_RANDOM_SEED,
) -> pd.DataFrame:
    if impact_episodes.empty or county_panel.empty:
        return pd.DataFrame()

    rng = np.random.default_rng(random_seed)
    panel_times_by_county = {
        county_fips: pd.DatetimeIndex(rows['time']).sort_values().unique()
        for county_fips, rows in county_panel.groupby('county_fips', sort=False)
    }
    impact_windows = [
        (row.start_time - exclusion_buffer, row.end_time + exclusion_buffer)
        for row in impact_episodes.itertuples()
    ]

    rows = []
    for episode in impact_episodes.itertuples():
        county_times = panel_times_by_county.get(str(episode.county_fips).zfill(5), pd.DatetimeIndex([]))
        if len(county_times) == 0:
            continue
        duration = pd.Timestamp(episode.end_time) - pd.Timestamp(episode.start_time)
        candidate_times = county_times[
            (county_times.month == pd.Timestamp(episode.start_time).month)
            & (county_times.dayofweek == pd.Timestamp(episode.start_time).dayofweek)
            & (county_times.hour == pd.Timestamp(episode.start_time).hour)
        ]
        for start, end in impact_windows:
            candidate_times = candidate_times[(candidate_times < start) | (candidate_times > end)]
        if len(candidate_times) == 0:
            continue
        selected = rng.choice(candidate_times.values, size=min(n_matches, len(candidate_times)), replace=False)
        for quiet_start in pd.DatetimeIndex(selected):
            rows.append({
                'impact_episode_id': episode.impact_episode_id,
                'county_fips': str(episode.county_fips).zfill(5),
                'quiet_start_time': quiet_start,
                'quiet_end_time': quiet_start + duration,
                'match_month': pd.Timestamp(episode.start_time).month,
                'match_dayofweek': pd.Timestamp(episode.start_time).dayofweek,
                'match_hour': pd.Timestamp(episode.start_time).hour,
                'match_duration_hours': float((duration + pd.Timedelta('15min')) / pd.Timedelta(hours=1)),
            })
    return pd.DataFrame(rows).sort_values(['impact_episode_id', 'quiet_start_time']).reset_index(drop=True)


if RUN_EAGLEI_QUIET_MATCHING:
    if impact_episodes.empty and EAGLEI_IMPACT_EPISODES_CSV.exists():
        impact_episodes = pd.read_csv(EAGLEI_IMPACT_EPISODES_CSV, parse_dates=['start_time', 'end_time', 'peak_time'], dtype={'county_fips': str})
    if county_outage_panel.empty and EAGLEI_COUNTY_PANEL_CSV.exists():
        county_outage_panel = pd.read_csv(EAGLEI_COUNTY_PANEL_CSV, parse_dates=['time'], dtype={'county_fips': str})
    quiet_periods = construct_time_matched_quiet_periods(impact_episodes, county_outage_panel)
    quiet_periods.to_csv(EAGLEI_QUIET_PERIODS_CSV, index=False)
    quiet_qa = pd.DataFrame([
        {'metric': 'impact_episodes', 'value': len(impact_episodes)},
        {'metric': 'quiet_periods', 'value': len(quiet_periods)},
        {'metric': 'episodes_with_matches', 'value': quiet_periods['impact_episode_id'].nunique() if not quiet_periods.empty else 0},
        {'metric': 'target_matches_per_episode', 'value': QUIET_MATCHES_PER_EPISODE},
    ])
    quiet_qa.to_csv(EAGLEI_QUIET_PERIOD_QA_CSV, index=False)
    display(quiet_qa)
else:
    quiet_periods = pd.read_csv(EAGLEI_QUIET_PERIODS_CSV, parse_dates=['quiet_start_time', 'quiet_end_time'], dtype={'county_fips': str}) if EAGLEI_QUIET_PERIODS_CSV.exists() else pd.DataFrame()
    print('Quiet-period matcher defined. Set RUN_EAGLEI_QUIET_MATCHING=True after impact episodes are cached.')
    print(f'Cached quiet periods loaded: {len(quiet_periods):,}')

## GeoE Hazard Episode Descriptors

For threshold discovery, treat geoE as an episode-level exposure rather than an instantaneous value. Candidate descriptors include magnitude, rate of change, duration, persistence above candidate thresholds, integrated exposure, and spatial coherence. These can be computed nationally, regionally, or after mapping counties/substations to analysis-grid cells.

In [ ]:
GEOE_DESCRIPTOR_COLUMNS = [
    'e_mag_max_mV_km',
    'e_mag_p95_mV_km',
    'e_mag_mean_mV_km',
    'd_e_dt_max_mV_km_per_min',
    'duration_hours_above_threshold',
    'max_consecutive_samples_above_threshold',
    'integrated_exposure_mV_km_hours',
    'max_active_area_fraction',
    'mean_active_area_fraction',
    'spatial_coherence_fraction',
]


def summarize_geoe_window(e_mag: xr.DataArray, start_time, end_time, candidate_threshold: float) -> dict:
    window = e_mag.sel(time=slice(pd.Timestamp(start_time), pd.Timestamp(end_time)))
    if window.sizes.get('time', 0) == 0:
        return {column: np.nan for column in GEOE_DESCRIPTOR_COLUMNS}

    values = np.asarray(window.values, dtype=float)
    finite = np.isfinite(values)
    active = finite & (values >= candidate_threshold)
    time_values = pd.DatetimeIndex(pd.to_datetime(window['time'].values))
    if len(time_values) >= 2:
        dt_hours = float(np.nanmedian(np.diff(time_values) / np.timedelta64(1, 'h')))
        dt_minutes = dt_hours * 60.0
    else:
        dt_hours = 0.0
        dt_minutes = np.nan

    spatial_max = np.nanmax(values.reshape(values.shape[0], -1), axis=1)
    d_e_dt = np.diff(spatial_max) / dt_minutes if np.isfinite(dt_minutes) and dt_minutes > 0 else np.array([])
    active_fraction = active.reshape(active.shape[0], -1).mean(axis=1)
    any_active = active.any(axis=(1, 2))

    max_run = 0
    current_run = 0
    for flag in any_active:
        current_run = current_run + 1 if flag else 0
        max_run = max(max_run, current_run)

    return {
        'e_mag_max_mV_km': float(np.nanmax(values)),
        'e_mag_p95_mV_km': float(np.nanpercentile(values[finite], 95)) if finite.any() else np.nan,
        'e_mag_mean_mV_km': float(np.nanmean(values)),
        'd_e_dt_max_mV_km_per_min': float(np.nanmax(np.abs(d_e_dt))) if d_e_dt.size else np.nan,
        'duration_hours_above_threshold': float(any_active.sum() * dt_hours),
        'max_consecutive_samples_above_threshold': int(max_run),
        'integrated_exposure_mV_km_hours': float(np.nansum(np.maximum(values - candidate_threshold, 0.0)) * dt_hours),
        'max_active_area_fraction': float(np.nanmax(active_fraction)) if active_fraction.size else np.nan,
        'mean_active_area_fraction': float(np.nanmean(active_fraction)) if active_fraction.size else np.nan,
        'spatial_coherence_fraction': float(np.nanmax(active_fraction)) if active_fraction.size else np.nan,
    }


print('GeoE descriptor helper ready. Load an interpolated |E| DataArray before applying it to impact and quiet windows.')

In [ ]:
def exceedance_shift_table(impact_values: pd.Series, quiet_values: pd.Series, thresholds: np.ndarray) -> pd.DataFrame:
    impact = pd.to_numeric(impact_values, errors='coerce').dropna().to_numpy()
    quiet = pd.to_numeric(quiet_values, errors='coerce').dropna().to_numpy()
    rows = []
    for threshold in thresholds:
        p_impact = (impact >= threshold).mean() if impact.size else np.nan
        p_quiet = (quiet >= threshold).mean() if quiet.size else np.nan
        rows.append({
            'threshold': threshold,
            'p_exceed_impact': p_impact,
            'p_exceed_quiet': p_quiet,
            'impact_conditioned_log_ratio': np.log((p_impact + 1e-6) / (p_quiet + 1e-6)),
            'risk_difference': p_impact - p_quiet,
        })
    return pd.DataFrame(rows)


def plot_exceedance_shift(shift_table: pd.DataFrame, title: str, output_path: Optional[Path] = None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(shift_table['threshold'], shift_table['p_exceed_impact'], label='impact episodes')
    axes[0].plot(shift_table['threshold'], shift_table['p_exceed_quiet'], label='matched quiet periods')
    axes[0].set_xlabel('candidate geoE threshold')
    axes[0].set_ylabel('exceedance probability')
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(shift_table['threshold'], shift_table['impact_conditioned_log_ratio'], color='black')
    axes[1].axhline(0, color='0.5', linewidth=0.8)
    axes[1].set_xlabel('candidate geoE threshold')
    axes[1].set_ylabel('log exceedance ratio')
    axes[1].grid(alpha=0.25)
    fig.suptitle(title)
    if output_path is not None:
        fig.savefig(output_path, dpi=150, bbox_inches='tight')
    return fig, axes


print('Threshold distribution-shift helpers ready.')

In [ ]:
# Eagle-I county-to-analysis-grid mapping
#
# Outputs from this cell are general Eagle-I processed mapping products, not specific to the
# space-weather threshold analysis. Space-weather-specific Eagle-I outputs use OUTPUT_DIR.

import zipfile
from urllib.request import urlretrieve

try:
    import geopandas as gpd
    from shapely.geometry import Point
except Exception as exc:
    raise ImportError('geopandas and shapely are required for county-to-grid mapping.') from exc

COUNTY_CACHE_DIR = EAGLEI_MAPPING_DIR / 'county_geometry_cache'
COUNTY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
COUNTY_ZIP_URL = 'https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip'
COUNTY_ZIP_PATH = COUNTY_CACHE_DIR / 'cb_2024_us_county_500k.zip'
COUNTY_SHP_PATH = COUNTY_CACHE_DIR / 'cb_2024_us_county_500k.shp'
COUNTY_GRID_MAPPING_CSV = EAGLEI_MAPPING_DIR / 'eaglei_county_to_analysis_grid_cells.csv'
COUNTY_INVENTORY_CSV = EAGLEI_MAPPING_DIR / 'eaglei_county_inventory.csv'


def eagle_i_csv_files_for_window(start_year: int = 2024, end_year: int = 2025) -> list[Path]:
    files = []
    for path in sorted(EAGLEI_DIR.rglob('*.csv')):
        if EAGLEI_PROCESSED_DIR in path.parents:
            continue
        name = path.name.lower()
        if any(str(year) in name for year in range(start_year, end_year + 1)):
            files.append(path)
    return files


def build_eaglei_county_inventory(paths: Iterable[Path], chunksize: int = 250_000) -> pd.DataFrame:
    rows = []
    for path in paths:
        header = pd.read_csv(path, nrows=0)
        fips_col = _find_column(header.columns, ['Fips Code', 'fips_code', 'county_fips', 'GEOID', 'fips'])
        county_col = _find_column(header.columns, ['County', 'county'])
        state_col = _find_column(header.columns, ['State', 'state'])
        if fips_col is None:
            print(f'Skipping {path.name}: no FIPS column found')
            continue
        usecols = [fips_col] + [column for column in [county_col, state_col] if column is not None]
        for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
            chunk = chunk.rename(columns={fips_col: 'county_fips'})
            if county_col is not None:
                chunk = chunk.rename(columns={county_col: 'county_name'})
            else:
                chunk['county_name'] = pd.NA
            if state_col is not None:
                chunk = chunk.rename(columns={state_col: 'state_abbr'})
            else:
                chunk['state_abbr'] = pd.NA
            chunk['county_fips'] = normalize_county_fips(chunk['county_fips'])
            chunk = chunk.dropna(subset=['county_fips'])
            rows.append(chunk[['county_fips', 'county_name', 'state_abbr']].drop_duplicates())
    if not rows:
        return pd.DataFrame(columns=['county_fips', 'county_name', 'state_abbr'])
    inventory = pd.concat(rows, ignore_index=True).drop_duplicates('county_fips')
    inventory = inventory.sort_values('county_fips').reset_index(drop=True)
    inventory['state_fips'] = inventory['county_fips'].str[:2]
    return inventory


def load_or_download_county_geometries() -> gpd.GeoDataFrame:
    if not COUNTY_SHP_PATH.exists():
        print(f'Downloading county geometries: {COUNTY_ZIP_URL}')
        urlretrieve(COUNTY_ZIP_URL, COUNTY_ZIP_PATH)
        with zipfile.ZipFile(COUNTY_ZIP_PATH) as archive:
            archive.extractall(COUNTY_CACHE_DIR)
    counties = gpd.read_file(COUNTY_SHP_PATH)
    counties['county_fips'] = counties['GEOID'].astype(str).str.zfill(5)
    counties['state_fips'] = counties['STATEFP'].astype(str).str.zfill(2)
    counties['county_name'] = counties['NAME'].astype(str)
    return counties[['county_fips', 'state_fips', 'county_name', 'geometry']].to_crs('EPSG:4326')


def build_grid_points(ds_grid: xr.Dataset) -> gpd.GeoDataFrame:
    lat = np.asarray(ds_grid['lat'].values, dtype=float)
    lon = np.asarray(ds_grid['lon'].values, dtype=float)
    y_idx, x_idx = np.indices(lat.shape)
    valid = np.isfinite(lat) & np.isfinite(lon)
    return gpd.GeoDataFrame(
        {
            'grid_y': y_idx[valid].astype(int),
            'grid_x': x_idx[valid].astype(int),
            'lat': lat[valid],
            'lon': lon[valid],
        },
        geometry=gpd.points_from_xy(lon[valid], lat[valid]),
        crs='EPSG:4326',
    )


def map_counties_to_grid_cells(counties: gpd.GeoDataFrame, grid_points: gpd.GeoDataFrame) -> pd.DataFrame:
    county_cols = ['county_fips', 'state_fips', 'county_name', 'geometry']
    joined = gpd.sjoin(grid_points, counties[county_cols], how='inner', predicate='within')
    rows = joined.drop(columns=['geometry', 'index_right']).copy()
    rows['mapping_method'] = 'grid_center_within_county'
    rows['distance_to_county_km'] = 0.0

    mapped_fips = set(rows['county_fips'].astype(str))
    missing = counties[~counties['county_fips'].isin(mapped_fips)].copy()
    if not missing.empty:
        grid_5070 = grid_points.to_crs('EPSG:5070')
        counties_5070 = missing.to_crs('EPSG:5070')
        grid_coords = np.column_stack([grid_5070.geometry.x, grid_5070.geometry.y])
        grid_tree = cKDTree(grid_coords)
        fallback_rows = []
        for county in counties_5070.itertuples():
            representative = county.geometry.representative_point()
            distance_m, grid_pos = grid_tree.query([representative.x, representative.y], k=1)
            grid_row = grid_points.iloc[int(grid_pos)]
            fallback_rows.append({
                'grid_y': int(grid_row['grid_y']),
                'grid_x': int(grid_row['grid_x']),
                'lat': float(grid_row['lat']),
                'lon': float(grid_row['lon']),
                'county_fips': county.county_fips,
                'state_fips': county.state_fips,
                'county_name': county.county_name,
                'mapping_method': 'nearest_grid_center_to_county_representative_point',
                'distance_to_county_km': float(distance_m / 1000.0),
            })
        rows = pd.concat([rows, pd.DataFrame(fallback_rows)], ignore_index=True)

    rows['county_fips'] = rows['county_fips'].astype(str).str.zfill(5)
    rows['state_fips'] = rows['state_fips'].astype(str).str.zfill(2)
    rows['county_grid_cell_count'] = rows.groupby('county_fips')['county_fips'].transform('size')
    return rows.sort_values(['county_fips', 'grid_y', 'grid_x']).reset_index(drop=True)


print(f'Eagle-I mapping outputs configured under: {EAGLEI_MAPPING_DIR}')

Eagle-I source files in run window: 369
Eagle-I counties with five-character FIPS: 3077
County geometries matched: 3069
Missing county geometries: 8
Grid cells mapped to Eagle-I counties: 4,142
Counties mapped: 3,069
Fallback nearest-cell county mappings: 1,154
Wrote county inventory: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_inventory.csv
Wrote county-grid mapping: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_to_analysis_grid_cells.csv


,grid_y,grid_x,lat,lon,county_fips,state_fips,county_name,mapping_method,distance_to_county_km,county_grid_cell_count
0,8,76,32.279518,-86.568084,01001,01,Autauga,nearest_grid_center_to_county_representative_p...,27.162628,1
1,4,74,30.573368,-87.814819,01003,01,Baldwin,grid_center_within_county,0.000000,2
2,5,74,31.021487,-87.769633,01003,01,Baldwin,grid_center_within_county,0.000000,2
3,7,78,31.739570,-85.565887,01005,01,Barbour,grid_center_within_county,0.000000,1
4,10,75,33.215111,-86.998306,01007,01,Bibb,nearest_grid_center_to_county_representative_p...,22.738869,1


In [10]:
# County-grid mapping QA: keep CONUS-supported mappings and recover Connecticut legacy counties

COUNTY_GRID_MAX_FALLBACK_DISTANCE_KM = 100.0
COUNTY_UNMAPPED_CSV = EAGLEI_SPACE_WEATHER_DIR / 'eaglei_counties_unmapped_to_analysis_grid.csv'
COUNTY_GRID_MAPPING_QA_CSV = EAGLEI_SPACE_WEATHER_DIR / 'eaglei_county_to_analysis_grid_cells_qa_summary.csv'
COUNTY_2023_ZIP_URL = 'https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip'
COUNTY_2023_ZIP_PATH = COUNTY_CACHE_DIR / 'cb_2023_us_county_500k.zip'
COUNTY_2023_SHP_PATH = COUNTY_CACHE_DIR / 'cb_2023_us_county_500k.shp'


def load_2023_county_geometries() -> gpd.GeoDataFrame:
    if not COUNTY_2023_SHP_PATH.exists():
        print(f'Downloading 2023 county geometries for legacy Connecticut county FIPS: {COUNTY_2023_ZIP_URL}')
        urlretrieve(COUNTY_2023_ZIP_URL, COUNTY_2023_ZIP_PATH)
        with zipfile.ZipFile(COUNTY_2023_ZIP_PATH) as archive:
            archive.extractall(COUNTY_CACHE_DIR)
    counties = gpd.read_file(COUNTY_2023_SHP_PATH)
    counties['county_fips'] = counties['GEOID'].astype(str).str.zfill(5)
    counties['state_fips'] = counties['STATEFP'].astype(str).str.zfill(2)
    counties['county_name'] = counties['NAME'].astype(str)
    return counties[['county_fips', 'state_fips', 'county_name', 'geometry']].to_crs('EPSG:4326')


def repair_county_grid_mapping(
    county_inventory: pd.DataFrame,
    mapping: pd.DataFrame,
    grid_points: gpd.GeoDataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    mapping = mapping.copy()
    mapping['county_fips'] = mapping['county_fips'].astype(str).str.zfill(5)
    mapping['state_fips'] = mapping['state_fips'].astype(str).str.zfill(2)

    invalid_far_fallback = (
        mapping['mapping_method'].ne('grid_center_within_county')
        & (mapping['distance_to_county_km'] > COUNTY_GRID_MAX_FALLBACK_DISTANCE_KM)
    )
    if invalid_far_fallback.any():
        print(f'Removing far nearest-cell fallbacks: {int(invalid_far_fallback.sum())} rows')
    mapping = mapping.loc[~invalid_far_fallback].copy()

    mapped_fips = set(mapping['county_fips'])
    missing_fips = sorted(set(county_inventory['county_fips'].astype(str).str.zfill(5)) - mapped_fips)

    if missing_fips:
        counties_2023 = load_2023_county_geometries()
        legacy_counties = counties_2023[counties_2023['county_fips'].isin(missing_fips)].copy()
        if not legacy_counties.empty:
            print(f'Recovering legacy county geometries from 2023 Census file: {len(legacy_counties)}')
            repaired = map_counties_to_grid_cells(legacy_counties, grid_points)
            repaired = repaired[
                repaired['mapping_method'].eq('grid_center_within_county')
                | (repaired['distance_to_county_km'] <= COUNTY_GRID_MAX_FALLBACK_DISTANCE_KM)
            ].copy()
            if not repaired.empty:
                mapping = pd.concat([mapping, repaired], ignore_index=True)

    mapping['county_fips'] = mapping['county_fips'].astype(str).str.zfill(5)
    mapping = mapping.drop_duplicates(['county_fips', 'grid_y', 'grid_x', 'mapping_method']).copy()
    mapping['county_grid_cell_count'] = mapping.groupby('county_fips')['county_fips'].transform('size')
    mapping = mapping.sort_values(['county_fips', 'grid_y', 'grid_x']).reset_index(drop=True)

    mapped_fips = set(mapping['county_fips'])
    unmapped = county_inventory[~county_inventory['county_fips'].isin(mapped_fips)].copy()
    unmapped['reason'] = 'outside_analysis_grid_or_no_supported_county_geometry'

    qa_summary = pd.DataFrame([
        {'metric': 'inventory_counties', 'value': int(county_inventory['county_fips'].nunique())},
        {'metric': 'mapped_counties', 'value': int(mapping['county_fips'].nunique())},
        {'metric': 'unmapped_counties', 'value': int(unmapped['county_fips'].nunique())},
        {'metric': 'mapping_rows', 'value': int(len(mapping))},
        {'metric': 'grid_center_rows', 'value': int(mapping['mapping_method'].eq('grid_center_within_county').sum())},
        {'metric': 'fallback_rows', 'value': int(mapping['mapping_method'].ne('grid_center_within_county').sum())},
        {'metric': 'max_fallback_distance_km', 'value': float(mapping.loc[mapping['mapping_method'].ne('grid_center_within_county'), 'distance_to_county_km'].max()) if mapping['mapping_method'].ne('grid_center_within_county').any() else 0.0},
    ])
    return mapping, unmapped, qa_summary


if 'county_grid_mapping' not in globals() or 'eaglei_county_inventory' not in globals() or 'grid_points' not in globals():
    county_grid_mapping = pd.read_csv(COUNTY_GRID_MAPPING_CSV, dtype={'county_fips': str, 'state_fips': str})
    eaglei_county_inventory = pd.read_csv(COUNTY_INVENTORY_CSV, dtype={'county_fips': str, 'state_fips': str})
    with xr.open_dataset(GRID_PATH) as ds_grid:
        grid_points = build_grid_points(ds_grid)

county_grid_mapping, unmapped_eaglei_counties, county_grid_mapping_qa = repair_county_grid_mapping(
    eaglei_county_inventory,
    county_grid_mapping,
    grid_points,
)
county_grid_mapping.to_csv(COUNTY_GRID_MAPPING_CSV, index=False)
unmapped_eaglei_counties.to_csv(COUNTY_UNMAPPED_CSV, index=False)
county_grid_mapping_qa.to_csv(COUNTY_GRID_MAPPING_QA_CSV, index=False)

print(f'Counties mapped after QA: {county_grid_mapping["county_fips"].nunique():,}')
print(f'Unmapped counties after QA: {unmapped_eaglei_counties["county_fips"].nunique():,}')
print(f'Max fallback distance after QA: {county_grid_mapping_qa.loc[county_grid_mapping_qa["metric"] == "max_fallback_distance_km", "value"].iloc[0]:.2f} km')
print(f'Rewrote county-grid mapping: {COUNTY_GRID_MAPPING_CSV}')
print(f'Wrote unmapped counties: {COUNTY_UNMAPPED_CSV}')
display(county_grid_mapping_qa)
if not unmapped_eaglei_counties.empty:
    display(unmapped_eaglei_counties.head(20))

Removing far nearest-cell fallbacks: 48 rows
Recovering legacy county geometries from 2023 Census file: 48
Counties mapped after QA: 3,021
Unmapped counties after QA: 56
Max fallback distance after QA: 93.16 km
Rewrote county-grid mapping: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_to_analysis_grid_cells.csv
Wrote unmapped counties: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_counties_unmapped_to_analysis_grid.csv


,metric,value
0,inventory_counties,3077.00000
1,mapped_counties,3021.00000
2,unmapped_counties,56.00000
3,mapping_rows,4094.00000
4,grid_center_rows,2988.00000
5,fallback_rows,1106.00000
6,max_fallback_distance_km,93.16077


,county_fips,county_name,state_abbr,state_fips,reason
66,02020,Anchorage,AK,02,outside_analysis_grid_or_no_supported_county_g...
67,02063,Chugach,AK,02,outside_analysis_grid_or_no_supported_county_g...
68,02068,Denali,AK,02,outside_analysis_grid_or_no_supported_county_g...
69,02090,Fairbanks North Star,AK,02,outside_analysis_grid_or_no_supported_county_g...
70,02122,Kenai Peninsula,AK,02,outside_analysis_grid_or_no_supported_county_g...
71,02170,Matanuska-Susitna,AK,02,outside_analysis_grid_or_no_supported_county_g...
72,02240,Southeast Fairbanks,AK,02,outside_analysis_grid_or_no_supported_county_g...
73,02290,Yukon-Koyukuk,AK,02,outside_analysis_grid_or_no_supported_county_g...
283,09001,Fairfield,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...
284,09003,Hartford,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...


In [11]:
# Preferred county-to-grid mapping: intersect Eagle-I county geometries with 50 km analysis-grid cells

from shapely.geometry import box

COUNTY_GRID_MAPPING_METHOD = 'grid_cell_intersects_county_polygon'


def build_grid_cell_polygons(ds_grid: xr.Dataset) -> gpd.GeoDataFrame:
    if 'x' not in ds_grid.coords or 'y' not in ds_grid.coords:
        raise ValueError('Analysis grid must include projected x/y coordinates to build cell polygons.')

    x_values = np.asarray(ds_grid['x'].values, dtype=float)
    y_values = np.asarray(ds_grid['y'].values, dtype=float)
    lat = np.asarray(ds_grid['lat'].values, dtype=float)
    lon = np.asarray(ds_grid['lon'].values, dtype=float)
    if x_values.ndim != 1 or y_values.ndim != 1:
        raise ValueError('Expected 1D projected x/y coordinates on the analysis grid.')

    dx = float(np.nanmedian(np.abs(np.diff(x_values))))
    dy = float(np.nanmedian(np.abs(np.diff(y_values))))
    y_idx, x_idx = np.indices(lat.shape)
    valid = np.isfinite(lat) & np.isfinite(lon)

    records = []
    geometries = []
    for yi, xi in zip(y_idx[valid].ravel(), x_idx[valid].ravel()):
        x0 = x_values[int(xi)]
        y0 = y_values[int(yi)]
        records.append({
            'grid_y': int(yi),
            'grid_x': int(xi),
            'lat': float(lat[int(yi), int(xi)]),
            'lon': float(lon[int(yi), int(xi)]),
        })
        geometries.append(box(x0 - dx / 2, y0 - dy / 2, x0 + dx / 2, y0 + dy / 2))

    return gpd.GeoDataFrame(records, geometry=geometries, crs='EPSG:5070')


def county_geometries_for_inventory(county_inventory: pd.DataFrame) -> gpd.GeoDataFrame:
    inventory_fips = set(county_inventory['county_fips'].astype(str).str.zfill(5))
    counties_2024 = load_or_download_county_geometries()
    counties_2024 = counties_2024[counties_2024['county_fips'].isin(inventory_fips)].copy()

    missing_fips = inventory_fips - set(counties_2024['county_fips'])
    pieces = [counties_2024]
    if missing_fips:
        counties_2023 = load_2023_county_geometries()
        legacy = counties_2023[counties_2023['county_fips'].isin(missing_fips)].copy()
        if not legacy.empty:
            pieces.append(legacy)
            print(f'Added legacy county geometries from 2023 Census file: {len(legacy)}')

    counties = pd.concat(pieces, ignore_index=True).drop_duplicates('county_fips')
    return gpd.GeoDataFrame(counties, geometry='geometry', crs='EPSG:4326')


def map_counties_to_grid_cell_polygons(
    counties: gpd.GeoDataFrame,
    grid_cells_5070: gpd.GeoDataFrame,
    grid_points: gpd.GeoDataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    counties_5070 = counties.to_crs('EPSG:5070')
    joined = gpd.sjoin(
        grid_cells_5070,
        counties_5070[['county_fips', 'state_fips', 'county_name', 'geometry']],
        how='inner',
        predicate='intersects',
    )
    mapping = joined.drop(columns=['geometry', 'index_right']).copy()
    mapping['mapping_method'] = COUNTY_GRID_MAPPING_METHOD
    mapping['distance_to_county_km'] = 0.0

    mapped_fips = set(mapping['county_fips'].astype(str))
    missing = counties_5070[~counties_5070['county_fips'].isin(mapped_fips)].copy()
    fallback_rows = []
    if not missing.empty:
        grid_points_5070 = grid_points.to_crs('EPSG:5070')
        grid_coords = np.column_stack([grid_points_5070.geometry.x, grid_points_5070.geometry.y])
        grid_tree = cKDTree(grid_coords)
        for county in missing.itertuples():
            representative = county.geometry.representative_point()
            distance_m, grid_pos = grid_tree.query([representative.x, representative.y], k=1)
            distance_km = float(distance_m / 1000.0)
            if distance_km > COUNTY_GRID_MAX_FALLBACK_DISTANCE_KM:
                continue
            grid_row = grid_points.iloc[int(grid_pos)]
            fallback_rows.append({
                'grid_y': int(grid_row['grid_y']),
                'grid_x': int(grid_row['grid_x']),
                'lat': float(grid_row['lat']),
                'lon': float(grid_row['lon']),
                'county_fips': county.county_fips,
                'state_fips': county.state_fips,
                'county_name': county.county_name,
                'mapping_method': 'nearest_grid_center_to_county_representative_point',
                'distance_to_county_km': distance_km,
            })
    if fallback_rows:
        mapping = pd.concat([mapping, pd.DataFrame(fallback_rows)], ignore_index=True)

    mapping['county_fips'] = mapping['county_fips'].astype(str).str.zfill(5)
    mapping['state_fips'] = mapping['state_fips'].astype(str).str.zfill(2)
    mapping = mapping.drop_duplicates(['county_fips', 'grid_y', 'grid_x', 'mapping_method']).copy()
    mapping['county_grid_cell_count'] = mapping.groupby('county_fips')['county_fips'].transform('size')
    mapping = mapping.sort_values(['county_fips', 'grid_y', 'grid_x']).reset_index(drop=True)

    unmapped = eaglei_county_inventory[~eaglei_county_inventory['county_fips'].isin(mapping['county_fips'])].copy()
    unmapped['reason'] = 'outside_analysis_grid_or_no_supported_county_geometry'
    return mapping, unmapped


if 'eaglei_county_inventory' not in globals():
    eaglei_county_inventory = pd.read_csv(COUNTY_INVENTORY_CSV, dtype={'county_fips': str, 'state_fips': str})
if 'grid_points' not in globals():
    with xr.open_dataset(GRID_PATH) as ds_grid:
        grid_points = build_grid_points(ds_grid)

with xr.open_dataset(GRID_PATH) as ds_grid:
    grid_cells_5070 = build_grid_cell_polygons(ds_grid)

counties_for_eaglei = county_geometries_for_inventory(eaglei_county_inventory)
county_grid_mapping, unmapped_eaglei_counties = map_counties_to_grid_cell_polygons(
    counties_for_eaglei,
    grid_cells_5070,
    grid_points,
)
county_grid_mapping_qa = pd.DataFrame([
    {'metric': 'inventory_counties', 'value': int(eaglei_county_inventory['county_fips'].nunique())},
    {'metric': 'mapped_counties', 'value': int(county_grid_mapping['county_fips'].nunique())},
    {'metric': 'unmapped_counties', 'value': int(unmapped_eaglei_counties['county_fips'].nunique())},
    {'metric': 'mapping_rows', 'value': int(len(county_grid_mapping))},
    {'metric': 'intersect_rows', 'value': int(county_grid_mapping['mapping_method'].eq(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'fallback_rows', 'value': int(county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'max_fallback_distance_km', 'value': float(county_grid_mapping.loc[county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD), 'distance_to_county_km'].max()) if county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).any() else 0.0},
])

county_grid_mapping.to_csv(COUNTY_GRID_MAPPING_CSV, index=False)
unmapped_eaglei_counties.to_csv(COUNTY_UNMAPPED_CSV, index=False)
county_grid_mapping_qa.to_csv(COUNTY_GRID_MAPPING_QA_CSV, index=False)

print(f'Counties mapped by grid-cell intersection: {county_grid_mapping["county_fips"].nunique():,}')
print(f'Unmapped counties: {unmapped_eaglei_counties["county_fips"].nunique():,}')
print(f'Mapping rows: {len(county_grid_mapping):,}')
print(f'Max fallback distance: {county_grid_mapping_qa.loc[county_grid_mapping_qa["metric"] == "max_fallback_distance_km", "value"].iloc[0]:.2f} km')
print(f'Wrote county-grid mapping: {COUNTY_GRID_MAPPING_CSV}')
print(f'Wrote unmapped counties: {COUNTY_UNMAPPED_CSV}')
display(county_grid_mapping_qa)
if not unmapped_eaglei_counties.empty:
    display(unmapped_eaglei_counties.head(20))

Counties mapped by grid-cell intersection: 3,021
Unmapped counties: 56
Mapping rows: 12,927
Max fallback distance: 93.16 km
Wrote county-grid mapping: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_to_analysis_grid_cells.csv
Wrote unmapped counties: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_counties_unmapped_to_analysis_grid.csv


,metric,value
0,inventory_counties,3077.00000
1,mapped_counties,3021.00000
2,unmapped_counties,56.00000
3,mapping_rows,12927.00000
4,intersect_rows,12914.00000
5,fallback_rows,13.00000
6,max_fallback_distance_km,93.16077


,county_fips,county_name,state_abbr,state_fips,reason
66,02020,Anchorage,AK,02,outside_analysis_grid_or_no_supported_county_g...
67,02063,Chugach,AK,02,outside_analysis_grid_or_no_supported_county_g...
68,02068,Denali,AK,02,outside_analysis_grid_or_no_supported_county_g...
69,02090,Fairbanks North Star,AK,02,outside_analysis_grid_or_no_supported_county_g...
70,02122,Kenai Peninsula,AK,02,outside_analysis_grid_or_no_supported_county_g...
71,02170,Matanuska-Susitna,AK,02,outside_analysis_grid_or_no_supported_county_g...
72,02240,Southeast Fairbanks,AK,02,outside_analysis_grid_or_no_supported_county_g...
73,02290,Yukon-Koyukuk,AK,02,outside_analysis_grid_or_no_supported_county_g...
283,09001,Fairfield,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...
284,09003,Hartford,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...


In [12]:
# Connecticut legacy county repair: Eagle-I uses pre-planning-region FIPS

COUNTY_2022_ZIP_URL = 'https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_county_500k.zip'
COUNTY_2022_ZIP_PATH = COUNTY_CACHE_DIR / 'cb_2022_us_county_500k.zip'
COUNTY_2022_SHP_PATH = COUNTY_CACHE_DIR / 'cb_2022_us_county_500k.shp'


def load_2022_county_geometries() -> gpd.GeoDataFrame:
    if not COUNTY_2022_SHP_PATH.exists():
        print(f'Downloading 2022 county geometries for Connecticut legacy county FIPS: {COUNTY_2022_ZIP_URL}')
        urlretrieve(COUNTY_2022_ZIP_URL, COUNTY_2022_ZIP_PATH)
        with zipfile.ZipFile(COUNTY_2022_ZIP_PATH) as archive:
            archive.extractall(COUNTY_CACHE_DIR)
    counties = gpd.read_file(COUNTY_2022_SHP_PATH)
    counties['county_fips'] = counties['GEOID'].astype(str).str.zfill(5)
    counties['state_fips'] = counties['STATEFP'].astype(str).str.zfill(2)
    counties['county_name'] = counties['NAME'].astype(str)
    return counties[['county_fips', 'state_fips', 'county_name', 'geometry']].to_crs('EPSG:4326')


if 'county_grid_mapping' not in globals():
    county_grid_mapping = pd.read_csv(COUNTY_GRID_MAPPING_CSV, dtype={'county_fips': str, 'state_fips': str})
if 'unmapped_eaglei_counties' not in globals():
    unmapped_eaglei_counties = pd.read_csv(COUNTY_UNMAPPED_CSV, dtype={'county_fips': str, 'state_fips': str})
if 'grid_points' not in globals():
    with xr.open_dataset(GRID_PATH) as ds_grid:
        grid_points = build_grid_points(ds_grid)
if 'grid_cells_5070' not in globals():
    with xr.open_dataset(GRID_PATH) as ds_grid:
        grid_cells_5070 = build_grid_cell_polygons(ds_grid)

legacy_missing_fips = sorted(
    set(unmapped_eaglei_counties.loc[unmapped_eaglei_counties['state_fips'].astype(str).str.zfill(2).eq('09'), 'county_fips'].astype(str).str.zfill(5))
)
print(f'Connecticut legacy county FIPS still unmapped before repair: {len(legacy_missing_fips)}')

if legacy_missing_fips:
    counties_2022 = load_2022_county_geometries()
    legacy_ct = counties_2022[counties_2022['county_fips'].isin(legacy_missing_fips)].copy()
    print(f'2022 legacy Connecticut county geometries found: {len(legacy_ct)}')
    if not legacy_ct.empty:
        legacy_mapping, _ = map_counties_to_grid_cell_polygons(legacy_ct, grid_cells_5070, grid_points)
        if not legacy_mapping.empty:
            county_grid_mapping = pd.concat([county_grid_mapping, legacy_mapping], ignore_index=True)
            county_grid_mapping['county_fips'] = county_grid_mapping['county_fips'].astype(str).str.zfill(5)
            county_grid_mapping['state_fips'] = county_grid_mapping['state_fips'].astype(str).str.zfill(2)
            county_grid_mapping = county_grid_mapping.drop_duplicates(['county_fips', 'grid_y', 'grid_x', 'mapping_method']).copy()
            county_grid_mapping['county_grid_cell_count'] = county_grid_mapping.groupby('county_fips')['county_fips'].transform('size')
            county_grid_mapping = county_grid_mapping.sort_values(['county_fips', 'grid_y', 'grid_x']).reset_index(drop=True)

mapped_fips = set(county_grid_mapping['county_fips'].astype(str).str.zfill(5))
unmapped_eaglei_counties = eaglei_county_inventory[~eaglei_county_inventory['county_fips'].isin(mapped_fips)].copy()
unmapped_eaglei_counties['reason'] = 'outside_analysis_grid_or_no_supported_county_geometry'
county_grid_mapping_qa = pd.DataFrame([
    {'metric': 'inventory_counties', 'value': int(eaglei_county_inventory['county_fips'].nunique())},
    {'metric': 'mapped_counties', 'value': int(county_grid_mapping['county_fips'].nunique())},
    {'metric': 'unmapped_counties', 'value': int(unmapped_eaglei_counties['county_fips'].nunique())},
    {'metric': 'mapping_rows', 'value': int(len(county_grid_mapping))},
    {'metric': 'intersect_rows', 'value': int(county_grid_mapping['mapping_method'].eq(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'fallback_rows', 'value': int(county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'max_fallback_distance_km', 'value': float(county_grid_mapping.loc[county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD), 'distance_to_county_km'].max()) if county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).any() else 0.0},
])
county_grid_mapping.to_csv(COUNTY_GRID_MAPPING_CSV, index=False)
unmapped_eaglei_counties.to_csv(COUNTY_UNMAPPED_CSV, index=False)
county_grid_mapping_qa.to_csv(COUNTY_GRID_MAPPING_QA_CSV, index=False)

print(f'Counties mapped after Connecticut repair: {county_grid_mapping["county_fips"].nunique():,}')
print(f'Unmapped counties after Connecticut repair: {unmapped_eaglei_counties["county_fips"].nunique():,}')
print(f'Connecticut legacy FIPS still unmapped: {unmapped_eaglei_counties[unmapped_eaglei_counties["state_fips"].astype(str).str.zfill(2).eq("09")]["county_fips"].nunique():,}')
print(f'Rewrote county-grid mapping: {COUNTY_GRID_MAPPING_CSV}')
display(county_grid_mapping_qa)
if not unmapped_eaglei_counties.empty:
    display(unmapped_eaglei_counties.head(20))

Connecticut legacy county FIPS still unmapped before repair: 8
2022 legacy Connecticut county geometries found: 0
Counties mapped after Connecticut repair: 3,021
Unmapped counties after Connecticut repair: 56
Connecticut legacy FIPS still unmapped: 8
Rewrote county-grid mapping: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_to_analysis_grid_cells.csv


,metric,value
0,inventory_counties,3077.00000
1,mapped_counties,3021.00000
2,unmapped_counties,56.00000
3,mapping_rows,12927.00000
4,intersect_rows,12914.00000
5,fallback_rows,13.00000
6,max_fallback_distance_km,93.16077


,county_fips,county_name,state_abbr,state_fips,reason
66,02020,Anchorage,AK,02,outside_analysis_grid_or_no_supported_county_g...
67,02063,Chugach,AK,02,outside_analysis_grid_or_no_supported_county_g...
68,02068,Denali,AK,02,outside_analysis_grid_or_no_supported_county_g...
69,02090,Fairbanks North Star,AK,02,outside_analysis_grid_or_no_supported_county_g...
70,02122,Kenai Peninsula,AK,02,outside_analysis_grid_or_no_supported_county_g...
71,02170,Matanuska-Susitna,AK,02,outside_analysis_grid_or_no_supported_county_g...
72,02240,Southeast Fairbanks,AK,02,outside_analysis_grid_or_no_supported_county_g...
73,02290,Yukon-Koyukuk,AK,02,outside_analysis_grid_or_no_supported_county_g...
283,09001,Fairfield,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...
284,09003,Hartford,Connecticut,09,outside_analysis_grid_or_no_supported_county_g...


In [13]:
# Connecticut legacy county repair using 2021 Census counties

COUNTY_2021_ZIP_URL = 'https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip'
COUNTY_2021_ZIP_PATH = COUNTY_CACHE_DIR / 'cb_2021_us_county_500k.zip'
COUNTY_2021_SHP_PATH = COUNTY_CACHE_DIR / 'cb_2021_us_county_500k.shp'


def load_2021_county_geometries() -> gpd.GeoDataFrame:
    if not COUNTY_2021_SHP_PATH.exists():
        print(f'Downloading 2021 county geometries for Connecticut legacy county FIPS: {COUNTY_2021_ZIP_URL}')
        urlretrieve(COUNTY_2021_ZIP_URL, COUNTY_2021_ZIP_PATH)
        with zipfile.ZipFile(COUNTY_2021_ZIP_PATH) as archive:
            archive.extractall(COUNTY_CACHE_DIR)
    counties = gpd.read_file(COUNTY_2021_SHP_PATH)
    counties['county_fips'] = counties['GEOID'].astype(str).str.zfill(5)
    counties['state_fips'] = counties['STATEFP'].astype(str).str.zfill(2)
    counties['county_name'] = counties['NAME'].astype(str)
    return counties[['county_fips', 'state_fips', 'county_name', 'geometry']].to_crs('EPSG:4326')


legacy_missing_fips = sorted(
    set(unmapped_eaglei_counties.loc[unmapped_eaglei_counties['state_fips'].astype(str).str.zfill(2).eq('09'), 'county_fips'].astype(str).str.zfill(5))
)
print(f'Connecticut legacy county FIPS still unmapped before 2021 repair: {len(legacy_missing_fips)}')

if legacy_missing_fips:
    counties_2021 = load_2021_county_geometries()
    legacy_ct = counties_2021[counties_2021['county_fips'].isin(legacy_missing_fips)].copy()
    print(f'2021 legacy Connecticut county geometries found: {len(legacy_ct)}')
    if not legacy_ct.empty:
        legacy_mapping, _ = map_counties_to_grid_cell_polygons(legacy_ct, grid_cells_5070, grid_points)
        if not legacy_mapping.empty:
            county_grid_mapping = pd.concat([county_grid_mapping, legacy_mapping], ignore_index=True)
            county_grid_mapping['county_fips'] = county_grid_mapping['county_fips'].astype(str).str.zfill(5)
            county_grid_mapping['state_fips'] = county_grid_mapping['state_fips'].astype(str).str.zfill(2)
            county_grid_mapping = county_grid_mapping.drop_duplicates(['county_fips', 'grid_y', 'grid_x', 'mapping_method']).copy()
            county_grid_mapping['county_grid_cell_count'] = county_grid_mapping.groupby('county_fips')['county_fips'].transform('size')
            county_grid_mapping = county_grid_mapping.sort_values(['county_fips', 'grid_y', 'grid_x']).reset_index(drop=True)

mapped_fips = set(county_grid_mapping['county_fips'].astype(str).str.zfill(5))
unmapped_eaglei_counties = eaglei_county_inventory[~eaglei_county_inventory['county_fips'].isin(mapped_fips)].copy()
unmapped_eaglei_counties['reason'] = 'outside_analysis_grid_or_no_supported_county_geometry'
county_grid_mapping_qa = pd.DataFrame([
    {'metric': 'inventory_counties', 'value': int(eaglei_county_inventory['county_fips'].nunique())},
    {'metric': 'mapped_counties', 'value': int(county_grid_mapping['county_fips'].nunique())},
    {'metric': 'unmapped_counties', 'value': int(unmapped_eaglei_counties['county_fips'].nunique())},
    {'metric': 'mapping_rows', 'value': int(len(county_grid_mapping))},
    {'metric': 'intersect_rows', 'value': int(county_grid_mapping['mapping_method'].eq(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'fallback_rows', 'value': int(county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).sum())},
    {'metric': 'max_fallback_distance_km', 'value': float(county_grid_mapping.loc[county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD), 'distance_to_county_km'].max()) if county_grid_mapping['mapping_method'].ne(COUNTY_GRID_MAPPING_METHOD).any() else 0.0},
])
county_grid_mapping.to_csv(COUNTY_GRID_MAPPING_CSV, index=False)
unmapped_eaglei_counties.to_csv(COUNTY_UNMAPPED_CSV, index=False)
county_grid_mapping_qa.to_csv(COUNTY_GRID_MAPPING_QA_CSV, index=False)

print(f'Counties mapped after 2021 Connecticut repair: {county_grid_mapping["county_fips"].nunique():,}')
print(f'Unmapped counties after 2021 Connecticut repair: {unmapped_eaglei_counties["county_fips"].nunique():,}')
print(f'Connecticut legacy FIPS still unmapped: {unmapped_eaglei_counties[unmapped_eaglei_counties["state_fips"].astype(str).str.zfill(2).eq("09")]["county_fips"].nunique():,}')
print(f'Rewrote county-grid mapping: {COUNTY_GRID_MAPPING_CSV}')
display(county_grid_mapping_qa)
if not unmapped_eaglei_counties.empty:
    display(unmapped_eaglei_counties.head(20))

Connecticut legacy county FIPS still unmapped before 2021 repair: 8
2021 legacy Connecticut county geometries found: 8
Counties mapped after 2021 Connecticut repair: 3,029
Unmapped counties after 2021 Connecticut repair: 48
Connecticut legacy FIPS still unmapped: 0
Rewrote county-grid mapping: /Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/eagle_i_space_weather/eaglei_county_to_analysis_grid_cells.csv


,metric,value
0,inventory_counties,3077.00000
1,mapped_counties,3029.00000
2,unmapped_counties,48.00000
3,mapping_rows,12959.00000
4,intersect_rows,12946.00000
5,fallback_rows,13.00000
6,max_fallback_distance_km,93.16077


,county_fips,county_name,state_abbr,state_fips,reason
66,02020,Anchorage,AK,02,outside_analysis_grid_or_no_supported_county_g...
67,02063,Chugach,AK,02,outside_analysis_grid_or_no_supported_county_g...
68,02068,Denali,AK,02,outside_analysis_grid_or_no_supported_county_g...
69,02090,Fairbanks North Star,AK,02,outside_analysis_grid_or_no_supported_county_g...
70,02122,Kenai Peninsula,AK,02,outside_analysis_grid_or_no_supported_county_g...
71,02170,Matanuska-Susitna,AK,02,outside_analysis_grid_or_no_supported_county_g...
72,02240,Southeast Fairbanks,AK,02,outside_analysis_grid_or_no_supported_county_g...
73,02290,Yukon-Koyukuk,AK,02,outside_analysis_grid_or_no_supported_county_g...
309,12011,Broward,FL,12,outside_analysis_grid_or_no_supported_county_g...
311,12015,Charlotte,FL,12,outside_analysis_grid_or_no_supported_county_g...
